<a href="https://colab.research.google.com/github/saskiaalifah/SaskiaAlifah_2411531002_BigData26/blob/main/Praktikum5/Praktikum5/BD_P05_2411531002_SaskiaAlifah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRAKTIKUM 5: PIPELINE DATA BERITA, DARI INGESTION HINGGA MAPREDUCE

In [ ]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

**Penjelasan kode.** Sel ini memasang pustaka yang dipakai praktikum: DuckDB (query SQL pada folder Parquet), BeautifulSoup4 dan lxml (parsing HTML dan sitemap XML), requests (HTTP), serta pyarrow dan pandas (membaca dan menulis Parquet). Opsi `-q` menekan pesan instalasi.

**Interpretasi hasil.** Tidak ada keluaran bila instalasi berhasil, karena mode `-q` hanya mencetak peringatan atau galat. Pustaka yang sudah terpasang di Colab (pandas, pyarrow, requests) tidak dipasang ulang. `lxml` wajib ada karena `BeautifulSoup(..., "xml")` pada H-4 memakai parser XML dari lxml.

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount("/content/drive")
    WORKDIR = Path("/content/drive/MyDrive/BigData/Praktikum5")
    WORKDIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORKDIR)
else:
    WORKDIR = Path.cwd()

print("Runtime Colab:", DI_COLAB)
print("Folder kerja :", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime Colab: True
Folder kerja : /content/drive/MyDrive/BigData/Praktikum5


**Penjelasan kode.** Sel ini mendeteksi apakah notebook berjalan di Google Colab (`from google.colab import drive`). Bila ya, Google Drive dipasang di `/content/drive`, folder kerja `MyDrive/BigData/Praktikum5` dibuat bila belum ada, dan `os.chdir()` memindahkan direktori kerja ke sana. Akibatnya semua jalur relatif pada sel berikutnya (`detik_news_title.csv`, `lapisan_mentah/`, `lapisan_terkurasi/`, `siap_pakai/`, `masukan.tsv`, dan arsip zip) otomatis tersimpan di Drive. Di luar Colab, direktori kerja saat ini dipakai.



In [ ]:
import platform
import pandas as pd, pyarrow, duckdb, requests, bs4, lxml.etree

print("Python        :", platform.python_version(), "|", platform.system())
print("pandas        :", pd.__version__)
print("pyarrow       :", pyarrow.__version__)
print("DuckDB        :", duckdb.__version__)
print("requests      :", requests.__version__)
print("beautifulsoup4:", bs4.__version__)
print("lxml          :", ".".join(map(str, lxml.etree.LXML_VERSION)))
print("CPU tersedia  :", os.cpu_count())

Python        : 3.13.18 | Linux
pandas        : 2.2.3
pyarrow       : 23.0.1
DuckDB        : 1.3.2
requests      : 2.32.4
beautifulsoup4: 4.13.5
lxml          : 6.1.3.0
CPU tersedia  : 2


**Penjelasan kode.** Sel ini mencetak versi Python dan pustaka yang benar-benar terpasang pada sesi ini, serta jumlah CPU yang tersedia.

**Interpretasi hasil.** Pada sesi Colab acuan terpasang Python 3.13, pandas 2.2.3, pyarrow 23.0.1, DuckDB 1.3.2, requests 2.32.4, beautifulsoup4 4.13.5, lxml 6.1.3, dengan **2 CPU**. Jumlah CPU menentukan makna eksperimen paralel pada H-10: dengan 2 CPU, `Pool(2)` benar-benar dapat memakai dua proses.

## H-1. Peta Pipeline dan Permainan Kartu Kantin

Pipeline praktikum ini terdiri dari enam tahap, dan setiap tahap menghasilkan data yang menjadi masukan tahap berikutnya:

| Tahap | Hasil | Langkah |
|---|---|---|
| Ingestion | `lapisan_mentah/` (JSONL) | H-3, H-4, H-5 |
| Quality dan cleaning | DataFrame bersih | H-6 |
| Storage | `lapisan_terkurasi/` (Parquet berpartisi) | H-6 |
| Query dan modeling | Tabel dan dokumen | H-7 |
| Processing (MapReduce) | Frekuensi kata | H-8 sampai H-12 |
| Serving | `siap_pakai/` (Parquet dan database) | H-13 |

**Permainan kartu kantin (dikerjakan tanpa laptop, hasilnya dicocokkan dengan kode di H-8).** Tiga kasir mencatat struknya masing-masing, pemilik ingin total porsi per menu.

| Kasir A | Kasir B | Kasir C |
|---|---|---|
| nasi goreng, 1 | nasi goreng, 2 | mie ayam, 2 |
| es teh, 2 | es teh, 1 | nasi goreng, 1 |
| mie ayam, 1 | es teh, 3 | es teh, 1 |
| es teh, 1 | mie ayam, 1 | nasi goreng, 1 |

1. **Map.** Tiap kasir menulis satu kartu per baris (menu, jumlah): 4 kartu per kasir, total **12 kartu**.
2. **Shuffle.** Semua kartu bermenu sama dikumpulkan ke satu meja: meja nasi goreng (1, 2, 1, 1), meja es teh (2, 1, 1, 3, 1), meja mie ayam (1, 1, 2).
3. **Reduce.** Setiap meja menjumlahkan kartunya: **nasi goreng 5, es teh 8, mie ayam 4** (total 17 porsi).
4. **Combiner.** Tiap kasir menjumlahkan dulu kartunya sendiri: Kasir A menjadi 3 kartu (nasi goreng 1, es teh 3, mie ayam 1), Kasir B 3 kartu (nasi goreng 2, es teh 4, mie ayam 1), Kasir C 3 kartu (mie ayam 2, nasi goreng 2, es teh 1). Kartu yang berpindah meja turun dari **12 menjadi 9**, dan hasil akhirnya tetap 5, 8, 4.



## H-2. Persiapan: Unduh Data dan Jalankan Portal Lokal

Dua sel berikut siap pakai. Sel pertama mengunduh CSV dataset judul berita detik.com dan melihat ukurannya. Sel kedua menyalakan portal berita mini di komputer sendiri yang melayani halaman `/berita?hal=N` (100 judul per halaman) dan berkas `robots.txt`.

In [ ]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))

Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']


**Penjelasan kode.** Sel ini mengimpor pustaka standar yang dipakai sepanjang notebook, mengunduh CSV dataset dari GitHub (hanya jika belum ada, sehingga aman dijalankan ulang), lalu membacanya ke `df_sumber` dan mencetak ukuran berkas, jumlah baris dan kolom, serta nama kolom.

**Interpretasi hasil.** Berkas berukuran **16,1 MB** dengan **91.017 baris dan 4 kolom** (`date`, `url`, `title`, `category`), persis sesuai deskripsi dataset. Angka 91.017 ini menjadi acuan pemeriksaan pertama pipeline: jumlah event yang masuk dan ditulis pada H-5 harus sama dengan angka ini. Format tanggal pada sumber adalah `bulan/hari/tahun` (misalnya `02/26/2020`), sehingga harus dikonversi dengan format eksplisit.

In [ ]:
import http.server, urllib.parse, html, socket

HALAMAN_UKURAN = 100  # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    # Izinkan penggunaan kembali alamat socket (reuse address)
    http.server.ThreadingHTTPServer.allow_reuse_address = True

    try:
        server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    except OSError:
        # Jika port 8000 benar-benar terkunci, gunakan port dinamis bebas (0)
        server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Handler)
        port = server.server_address[1]

    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal, port

server, N_HALAMAN, PORT_AKTIF = jalankan_portal(df_sumber)
BASE = f"http://127.0.0.1:{PORT_AKTIF}"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:34557 | jumlah halaman: 911


**Penjelasan kode.** Sel ini menyalakan portal berita mini di komputer sendiri (`127.0.0.1:8000`) pada thread terpisah. Portal menyajikan `/berita?hal=N` (100 judul per halaman, tiap judul dibungkus `<article class="item">` lengkap dengan kategori dan tanggal ISO pada atribut `datetime`), `/robots.txt` yang melarang `/arsip-privat/`, dan halaman 404 untuk alamat lain. Isi sel ini tidak perlu dipahami rinci; cukup tahu bahwa ia menyajikan isi CSV sebagai HTML agar latihan crawling tidak membebani situs sungguhan.

**Interpretasi hasil.** Portal aktif di `http://127.0.0.1:8000` dengan **911 halaman**, yaitu 91.017 judul dibagi 100 judul per halaman dan dibulatkan ke atas (910 halaman penuh ditambah 1 halaman berisi 17 judul). Angka 911 dipakai lagi sebagai jumlah halaman yang di-crawl pada H-5.

## H-3. Crawling Portal Lokal: Latihan Aman

Portal lokal selalu tersedia dan hasilnya sama untuk semua orang, sehingga cocok untuk berlatih mekanisme crawling sebelum menyentuh situs sungguhan di H-4. Langkah pertama crawler yang baik adalah membaca `robots.txt`. Sel berikut memeriksa dua alamat: halaman berita biasa dan `/arsip-privat/` yang dilarang oleh portal.

In [ ]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


**Penjelasan kode.** `baca_robots()` mengunduh `robots.txt` sebuah host dan mengubahnya menjadi objek `RobotFileParser`. Crawler memperkenalkan diri lewat `User-Agent`, lalu `can_fetch()` ditanyakan untuk dua alamat: halaman berita biasa dan `/arsip-privat/`.

**Interpretasi hasil.** `/berita?hal=1` **boleh** di-crawl (`True`) sedangkan `/arsip-privat/` **tidak boleh** (`False`), sesuai aturan `Disallow: /arsip-privat/` pada portal. Ini langkah pertama crawler yang bertanggung jawab: aturan situs dibaca dan dipatuhi sebelum satu halaman pun diambil.

Berikutnya, `parse_halaman` mengubah HTML menjadi data (menentukan elemen mana yang berisi judul, kategori, dan tanggal). Fungsi `crawl_iter` memeriksa `robots.txt`, mencoba ulang bila gagal, dan memberi jeda antar-halaman.

In [ ]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


**Penjelasan kode.** `parse_halaman()` memakai CSS selector: `article.item` memilih tiap judul, `h2.judul a` memberi judul dan tautan, `span.kategori` memberi kategori, dan atribut `datetime` pada `<time>` memberi tanggal ISO. `crawl_iter()` adalah generator yang memeriksa `robots.txt`, mencoba ulang tiap halaman sampai 3 kali bila gagal, memberi jeda antar-halaman, dan mengembalikan hasil satu demi satu lewat `yield from`. Sel ini mencoba tiga halaman pertama.

**Interpretasi hasil.** Tiga halaman menghasilkan **300 judul** (3 halaman x 100 judul). Tiga contoh pertama berupa dictionary berisi `judul`, `url`, `kategori`, dan `tanggal` (format `2020-02-26`, kategori `finance`). Tanggal diambil dari atribut `datetime` (format ISO) dan bukan dari teks `02/26/2020` karena teks tersebut bisa ambigu antara bulan/hari dan hari/bulan.

## H-4. Crawling Berita Sungguhan dari Sitemap detik.com

Sekarang data diambil dari sumber sebenarnya. Halaman artikel tidak dibuka satu per satu karena sitemap berita detik.com sudah memuat judul, tanggal terbit, dan kata kunci. **Langkah 1** menulis parser `parse_sitemap_news` (bagian kunci): tag `title` dan `publication_date` dicari tanpa awalan namespace `news:`, kategori diambil dari subdomain URL, id dari pola `/d-<angka>/`, dan tanggal hanya 10 karakter pertama (`YYYY-MM-DD`) karena tanda zona waktu pada sitemap tampak keliru.

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else urlparse(url).netloc.split(".")[0],  # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

**Penjelasan kode.** `parse_sitemap_news()` mengubah satu berkas sitemap berita (XML) menjadi daftar dictionary. Untuk tiap `<url>`, `find("title")` dan `find("publication_date")` menemukan `news:title` dan `news:publication_date` karena awalan namespace diabaikan saat pencarian tag. Kategori diambil dari subdomain URL lewat `POLA_ARTIKEL` (`news`, `inet`, `sport`), id dari pola `/d-<angka>/`, dan tanggal dipotong 10 karakter pertama karena tanda zona waktu pada sitemap tampak keliru. Entri tanpa `loc` atau judul dilewati agar satu data cacat tidak menghentikan seluruh proses.

**Interpretasi hasil.** Sel ini hanya mendefinisikan fungsi sehingga tidak ada keluaran. Fungsi diuji pada sel berikutnya.

**Langkah 2: uji tanpa internet.** Sebelum menghubungi server, parser diuji pada contoh XML kecil yang disusun dari tiga entri nyata hasil uji dosen (29 Oktober 2026). Kata kunci hanya tersedia pada entri pertama.

In [ ]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


**Penjelasan kode.** Parser diuji pada `CONTOH_XML` (tiga entri nyata hasil uji dosen, 29 Oktober 2026) tanpa menghubungi server. Pengujian mencetak judul, kategori, id, dan tanggal tiap entri, kata kunci entri pertama, serta membandingkan hasil parsing masukan `str` dengan masukan `bytes`.

**Interpretasi hasil.** Parser mengembalikan **3 entri** dengan kategori `news`, id **8729282**, **8729280**, dan **8729264**, serta tanggal `2026-10-29`. Judul yang dibungkus CDATA terbaca sebagai teks biasa, dan judul yang terpotong baris diratakan oleh `re.sub(r"\s+", " ", ...)`. Kata kunci hanya ada pada entri pertama (`mahkamah konstitusi,mk,hukum,kepala daerah`), sedangkan dua entri lain tidak memilikinya sehingga bernilai kosong. Hasil parsing `str` dan `bytes` **sama** (`True`), yang penting karena crawler nanti memakai `r.content` (bytes) agar pengkodean dibaca dari deklarasi XML dan tidak ditebak.

**Langkah 3: fungsi pengambilan yang sopan.** Ada enam pengaman: `robots.txt` dibaca per host, hanya tiga sitemap yang diambil (`SITEMAP_DIPILIH`), maksimum tujuh judul per sitemap dan 20 judul total, jeda 2 detik antar-request, User-Agent yang menyebut identitas, dan berhenti seketika bila server membalas 403 atau 429.

In [ ]:
import urllib.robotparser, socket
socket.setdefaulttimeout(20)   # tambahan pengaman: robots.txt dibaca lewat urllib yang tidak punya batas waktu bawaan

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: saskiaalifah2006@gmail.com)"
DELAY = 2.0              # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7     # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20         # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

**Penjelasan kode.** Sel ini mendefinisikan enam pengaman crawler: (1) `boleh_diambil()` membaca `robots.txt` per host dan menyimpannya di `_robots` supaya tiap host hanya diunduh sekali, (2) hanya tiga sitemap pada `SITEMAP_DIPILIH`, (3) maksimum 7 judul per sitemap (`BATAS_PER_BAGIAN`), (4) maksimum 20 judul total (`BATAS_TOTAL`), (5) jeda 2 detik antar-request (`DELAY`), dan (6) `ambil_xml()` berhenti dengan galat bila server membalas 403 atau 429 agar crawler tidak mendesak server yang sudah menolak.

**Interpretasi hasil.** Sel ini hanya mendefinisikan fungsi dan konstanta sehingga tidak ada keluaran, dan tidak ada request yang dikirim sampai `crawl_detik()` dipanggil pada sel berikutnya.

**Langkah 4: jalankan dan periksa.** Hasil disimpan di folder `contoh_crawl/`, terpisah dari `lapisan_mentah/` agar angka acuan pada langkah berikutnya tidak berubah. Sel ini hanya dijalankan **sekali** (jangan diulang dan jangan menaikkan `BATAS_TOTAL`), karena laboratorium sering memakai satu alamat IP publik bersama.

In [ ]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul :", len(cek))
    print("Judul kosong    :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat    :", cek["url"].duplicated().sum())
    print("Tanggal kosong  :", cek["tanggal"].isna().sum())
    print("Kategori        :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul : 20
Judul kosong    : 0
URL duplikat    : 0
Tanggal kosong  : 0
Kategori        : {'news': 7, 'inet': 7, 'sport': 6}
                                                          judul kategori    tanggal
Gibran Resmi mengundurkan Diri, PDIP: Momentum Koreksi Nasional     news 2026-11-12
                                                   Gibran Pamit     news 2026-11-12
                Momen Gibran Temui Prabowo Sebelum Resmi Mundur     news 2026-11-12


**Penjelasan kode.** `crawl_detik()` dijalankan di dalam `try/except` sehingga kegagalan jaringan tidak menghentikan notebook. Bila berhasil, 20 judul disimpan sebagai JSONL di `contoh_crawl/detik_sitemap.jsonl` (terpisah dari `lapisan_mentah/`), lalu diperiksa: jumlah judul, judul kosong, URL duplikat, tanggal kosong, dan sebaran kategori.

**Interpretasi hasil.** Pada sesi Colab acuan, crawl berhasil: **20 judul**, **0 judul kosong**, **0 URL duplikat**, **0 tanggal kosong**, dengan kategori `{'news': 7, 'inet': 7, 'sport': 6}` (7 + 7 + 6). Tiga contoh judul yang tercetak berbeda setiap hari (tanggalnya adalah tanggal crawl), sehingga isi tidak punya angka acuan tetap; yang diperiksa adalah bentuknya. Bila yang tercetak `Crawling gagal: ...`, penyebabnya koneksi ke detik.com atau larangan `robots.txt`, dan notebook tetap berjalan karena `sampel` dikosongkan. Sel ini hanya dijalankan sekali agar tidak membebani server.

## H-5. Channel ala Flume ke lapisan_mentah

Untuk skala penuh, seluruh 911 halaman portal lokal di-crawl (bukan detik.com, yang tidak boleh dibebani 91 ribu judul sekaligus). Crawler adalah **source**, antrean `queue.Queue` adalah **channel** (maksimal 2.000 event), dan fungsi **sink** menulis setiap 5.000 event menjadi satu berkas JSONL di `lapisan_mentah/`. Source dan sink berjalan di thread terpisah, dan bila channel penuh, source otomatis menunggu (backpressure).

In [ ]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)                      # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 57.5 detik untuk 911 halaman


**Penjelasan kode.** Crawler berperan sebagai **source** yang meng-crawl 911 halaman portal lokal dan memasukkan tiap judul ke **channel** (`queue.Queue` berkapasitas 2.000 event; `put()` menunggu bila penuh). **Sink** mengambil event dari channel dan menulis tiap 5.000 event menjadi satu berkas `batch_XXXX.jsonl` di `lapisan_mentah/`. Keduanya berjalan di thread terpisah, dan objek `STOP` menandai akhir data. Statistik dihitung untuk membuktikan tidak ada data hilang di tengah jalan.

**Interpretasi hasil.** Event yang masuk channel dan event yang ditulis sink sama-sama **91.017**, sama dengan jumlah baris CSV, sehingga tidak ada data hilang. Terbentuk **19 berkas batch** (18 berkas berisi 5.000 event dan 1 berkas berisi sisa 1.017), `Antrean maks <= 2000` bernilai `True` (backpressure bekerja; antrean maksimum yang tercapai bervariasi antar-eksekusi, ratusan event pada sesi acuan), dan total ukuran mentah sekitar **20,7 MB** (JSONL lebih besar daripada CSV 16,1 MB karena nama kolom diulang di setiap baris).

## H-6. Quality, Cleaning, dan Parquet Berpartisi

Seperti pada praktikum sebelumnya, data diperiksa dulu sebelum dibersihkan. Fungsi `aturan_kualitas` (bagian kunci) berisi enam aturan, dan setiap aturan menghasilkan nilai benar/salah per baris: judul 3 sampai 30 kata, kategori termasuk sembilan kategori yang dikenal, dan URL artikel yang valid (memuat pola `/d-<angka>`).

In [ ]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"  : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "kategori dikenal"    : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"       : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"   : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


**Penjelasan kode.** Seluruh berkas JSONL di `lapisan_mentah/` dibaca dan digabung menjadi `mentah`. `aturan_kualitas()` berisi enam aturan, masing-masing menghasilkan Series benar/salah per baris: judul tidak kosong, judul 3 sampai 30 kata, kategori termasuk sembilan kategori yang dikenal, tanggal valid dengan format `%Y-%m-%d`, URL artikel valid (memuat `/d-<angka>`), dan judul tidak duplikat (`keep=False` menandai semua salinan). Jumlah dan persentase baris yang gagal dicetak tiap aturan.

**Interpretasi hasil.** Baris mentah **91.017**. Hasil pemeriksaan: judul kosong 0, judul di luar 3 sampai 30 kata **78 baris (0,09%)**, kategori tidak dikenal 0, tanggal tidak valid 0, URL artikel tidak valid **5.640 baris (6,20%)**, dan judul duplikat **235 baris (0,26%)** (semua salinan yang ditandai, bukan hanya salinan kedua dan seterusnya). Temuan kualitas yang paling penting: **5.640 baris yang gagal aturan URL persis sama dengan seluruh baris kategori `inet`**, yang URL-nya bernilai `#`; ini cacat dari scraper pembuat dataset, bukan galat pipeline kita (dibahas pada L-4).

In [ ]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")   # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


**Penjelasan kode.** Pembersihan berurutan: spasi dirapikan, judul duplikat dibuang dengan mempertahankan yang pertama, judul di luar 3 sampai 30 kata dibuang, tanggal diubah ke `datetime` dengan format eksplisit, lalu dibuat kolom `bulan` (`YYYY-MM`) dan `id` (diekstrak dari URL, bertipe `Int64` agar boleh kosong). URL yang rusak dikosongkan lewat `where`, tetapi barisnya **tidak dibuang**.

**Interpretasi hasil.** Dari 91.017 baris tersisa **90.818 baris**, dan **199 baris dibuang** (0,22%): judul duplikat (salinan kedua dan seterusnya) dan judul yang terlalu pendek atau panjang. Baris `inet` tetap ada karena judulnya masih berguna untuk analisis teks, sehingga satu kategori utuh tidak hilang dari analisis; yang dikosongkan hanya kolom `url` dan `id`-nya. Pilihan antara membuang, memperbaiki, atau mempertahankan baris cacat adalah keputusan analisis, dan alasannya dicatat di sini.

Terakhir, data disimpan sebagai Parquet berpartisi. **Bagian kunci:** memilih kolom partisi. Kode di bawah mencoba tiga pilihan dan mencetak jumlah berkasnya, lalu menyimpan versi final (partisi `kategori`) dan mendemonstrasikan *partition pruning*.

In [ ]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan tiga pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


**Penjelasan kode.** `simpan_terpartisi()` menghapus folder lama (agar idempoten), menulis `bersih` sebagai Parquet dengan `partition_cols`, lalu mengembalikan jumlah berkas dan total ukuran. **Bagian kunci:** tiga pilihan kolom partisi dibandingkan (`kategori`, `bulan`, dan `kategori + bulan`), lalu versi final disimpan ke `lapisan_terkurasi/` dengan partisi `kategori`. Terakhir, `pyarrow.dataset` dipakai untuk menunjukkan *partition pruning*: hanya berkas yang sesuai filter yang dibaca.

**Interpretasi hasil.** Partisi `kategori` menghasilkan **9 berkas (sekitar 8,4 MB)**, partisi `bulan` **6 berkas (sekitar 8,3 MB)**, dan partisi `kategori + bulan` **54 berkas (sekitar 8,4 MB)**. Ukuran total hampir sama, jadi 54 berkas hanya berarti banyak berkas kecil tanpa keuntungan (*small file problem*). Dipilih partisi **kategori** karena jumlahnya sedikit, sesuai pertanyaan analisis (perbedaan topik antar-kategori), dan nanti dipakai sebagai input split MapReduce. *Partition pruning*: untuk `kategori = finance`, hanya **1 dari 9 berkas** yang dibaca.

## H-7. Query DuckDB dan Bentuk Dokumen

Lapisan terkurasi dibaca dengan DuckDB. Kolom `kategori` dibaca dari nama folder (*hive partitioning*). Query kedua menghitung judul yang memuat kata "corona" per bulan. Terakhir, satu baris diubah menjadi **dokumen**: bentuk fleksibel yang bisa menyimpan daftar kata di dalam satu record.

In [ ]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


**Penjelasan kode.** Lapisan terkurasi dibaca DuckDB lewat view `berita` (`read_parquet(..., hive_partitioning=true)`), sehingga kolom `kategori` diambil dari nama folder. Query pertama menghitung judul per kategori, query kedua menghitung judul yang memuat "corona" per bulan. Terakhir, satu baris diubah menjadi **dokumen** (dictionary dengan daftar `kata` di dalam satu record).

**Interpretasi hasil.** Jumlah judul per kategori: `news` 32.321, `hot` 16.290, `finance` 14.141, `travel` 6.453, `inet` 5.577, `health` 4.914, `oto` 4.375, `food` 4.314, `sport` 2.433 (total 90.818, sama dengan data bersih). Judul memuat "corona": Januari 385, Februari 1.077, **Maret 6.994 (puncak)**, April 4.913, Mei 2.934, dan Juni 943 (hanya 13 hari pertama, sehingga tidak sebanding dengan bulan penuh). Dokumen hasil: `id` 4916114, `judul` "Kemnaker Awasi TKA di Meikarta", kategori `finance`, tanggal `2020-02-26`, dan `kata` berisi lima kata. Bentuk tabel cocok untuk menghitung dan mengelompokkan, sedangkan bentuk dokumen cocok untuk menyimpan satu artikel beserta atribut turunannya (daftar kata atau tag) tanpa mengubah skema tabel.

## H-8. Mini MapReduce: Kasus Kantin

Sebelum memproses 90 ribu judul, kasus kantin dikerjakan dalam kode. Tiga fungsi (`map_fn`, `shuffle`, `reduce_fn`) mewakili tiga tahap, dan setiap tahap dicetak agar terlihat bagaimana data berubah bentuk.

In [ ]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner :", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner : 12
Pasangan dikirim dengan combiner: 9


**Penjelasan kode.** Kasus kantin dikerjakan dalam tiga fungsi: `map_fn` mengubah satu baris `"menu,jumlah"` menjadi pasangan `(menu, jumlah)`, shuffle mengumpulkan nilai berdasarkan key ke dalam dictionary, dan `reduce_fn` menjumlahkan nilai per key. `combiner()` menjumlahkan dulu pasangan di setiap kasir (sisi mapper) sebelum dikirim.

**Interpretasi hasil.** Hasil reduce adalah **nasi goreng 5, es teh 8, mie ayam 4** (total 17 porsi), persis sama dengan permainan kartu di H-1. Tahap shuffle mengumpulkan `nasi goreng: [1, 2, 1, 1]`, `es teh: [2, 1, 1, 3, 1]`, dan `mie ayam: [1, 1, 2]`. Pasangan yang dikirim ke reducer turun dari **12 menjadi 9** dengan combiner, tanpa mengubah hasil akhir.

## H-9. Word Count Terdistribusi: Partisi sebagai Input Split

Pola yang sama diterapkan pada judul berita. Satu **input split** adalah satu berkas partisi Parquet, jadi ada 9 mapper. Cara data dipartisi menentukan cara pekerjaan dibagi, dan ukuran split sangat tidak seimbang (kategori `news` memiliki 32 ribu judul, `sport` hanya 2 ribu).

In [ ]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")

print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


**Penjelasan kode.** Pola kantin diterapkan pada judul berita. Satu **input split** adalah satu berkas partisi Parquet, jadi ada satu mapper per kategori. `mapper_split()` membaca satu split dan mengembalikan pasangan `(kata, 1)` dari tiap kata hasil `tokenisasi()`. Hasil semua mapper di-shuffle (dikelompokkan per kata) lalu direduce dengan `sum`. Hasilnya dicek silang dengan `collections.Counter` biasa tanpa MapReduce.

**Interpretasi hasil.** Terdapat **9 input split** dengan ukuran sangat tidak seimbang (`news` 32.321 judul, `sport` hanya 2.433). Semua mapper menghasilkan **825.601 pasangan** dengan **32.275 kata unik** (key setelah shuffle). Sepuluh kata teratas dengan stopword masih didominasi kata fungsi (`di` 27.390, `corona` 17.239, `ini` 10.124, `yang` 8.668, `dan` 6.785, ...), sedangkan setelah stopword dibuang kata teratas adalah **corona (17.239)**, `rp` (3.154), `virus` (3.075), `kasus` (2.743), `polisi` (2.732), `positif`, `indonesia`, `warga`, `rumah`, dan `jokowi`. Hasil MapReduce **sama dengan Counter biasa** (`True`). Daftar stopword hanya berisi kata fungsi umum sehingga `rp` tetap muncul; menentukan stopword untuk domain tertentu adalah keputusan analisis (dibahas pada L-3).

## H-10. Combiner dan Jumlah Data yang Berpindah

Pertanyaannya: apakah menjalankan mapper di beberapa proses membuat pekerjaan lebih cepat, dan apa pengaruh combiner? Sel berikut mengukur waktu (median dari tiga kali jalan) dan menghitung pasangan yang dikirim ke reducer. Angka pasangan stabil, sedangkan waktu bervariasi antar-komputer dan antar-kali-jalan.

In [ ]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.605              825,601
1 proses, dengan combiner         0.661               74,009
2 proses, tanpa combiner          2.566              825,601
2 proses, dengan combiner         0.901               74,009

Hasil dengan combiner sama persis: True


**Penjelasan kode.** `mapper_combiner()` menghitung kata dengan `Counter` di sisi mapper sehingga hanya mengembalikan satu pasangan per kata unik per mapper. `jalankan()` menjalankan mapper pada satu proses atau pada `Pool` beberapa proses, dan mengukur waktunya. Empat skenario (1 atau 2 proses, tanpa atau dengan combiner) dijalankan tiga kali dan dicetak waktu mediannya beserta jumlah pasangan yang dikirim ke reducer.

**Interpretasi hasil.** Angka pasangan bersifat **tetap**: **825.601** tanpa combiner dan **74.009** dengan combiner, berkurang sekitar **11 kali lipat** dengan hasil akhir yang persis sama (`True`).  Pada 1 proses, combiner sedikit lebih lambat atau setara karena ada biaya `Counter` di sisi mapper: combiner bukan untuk mempercepat mapper, melainkan untuk mengurangi data yang harus dipindahkan. Pelajarannya, paralel tidak otomatis lebih cepat dan yang mahal adalah memindahkan data (dibuktikan lebih lanjut pada L-1).

## H-11. Data Skew dan Beban Reducer

Partitioner memutuskan reducer mana yang menerima sebuah key. Hash yang dipakai adalah CRC32. Sel berikut menghitung beban setiap reducer, lalu membandingkan reducer terberat dengan bagian yang adil bila jumlah reducer ditambah.

In [ ]:
R = 4   # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


**Penjelasan kode.** `partitioner()` memetakan sebuah kata ke reducer lewat `CRC32(kata) % R`, yang hasilnya sama di setiap komputer (fungsi `hash()` Python tidak dipakai karena berubah antar-sesi). `beban()` menghitung berapa pasangan yang diterima tiap reducer. Sel ini mencetak beban empat reducer tanpa dan dengan combiner, membandingkan reducer terberat dengan bagian yang adil untuk 4, 16, dan 64 reducer, dan menunjukkan beban lima key terbesar.

**Interpretasi hasil.** Tanpa combiner, empat reducer menerima 29,2%, 20,6%, 23,1%, dan 27,1% pasangan (R0 terberat), sedangkan dengan combiner hampir rata: 24,7%, 24,7%, 25,0%, dan 25,6%. Reducer terberat dibanding bagian yang adil: **29,17% vs 25,00%** (4 reducer), **8,82% vs 6,25%** (16 reducer), dan **4,28% vs 1,56%** (64 reducer), hampir tiga kali lipat. Penyebabnya, kata `di` sendirian **3,3%** dari seluruh pasangan (27.390 pasangan, di reducer R0), dan satu key tidak bisa dipecah ke banyak reducer, sehingga menambah reducer tidak menyelesaikan skew yang berasal dari satu key. Combiner membantu karena setiap kata hanya dikirim sekali per mapper.

## H-12. Hadoop Streaming

Hadoop Streaming menjalankan mapper dan reducer sebagai program biasa yang membaca `stdin` dan menulis `stdout`. Sel pertama menulis berkas masukan (kategori dan judul dipisahkan tab) serta dua skrip `mapper.py` dan `reducer.py`.

In [ ]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


**Penjelasan kode.** Sel ini menulis tiga berkas: `masukan.tsv` (kategori dan judul dipisahkan tab, satu baris per judul bersih), `mapper.py` (membaca `stdin`, mengambil kolom judul, dan mencetak `kata<TAB>1` untuk tiap kata), dan `reducer.py` (membaca pasangan yang sudah terurut, menjumlahkan nilai selama key sama, dan mencetak total saat key berganti). Tanda `\\n` dan `\\t` ditulis dengan dua garis miring terbalik agar menjadi `\n` dan `\t` di dalam berkas skrip.

**Interpretasi hasil.** `masukan.tsv` berisi **90.818 baris**, sama dengan jumlah baris data bersih pada H-6, jadi tidak ada baris yang hilang saat diekspor ke bentuk teks. Reducer mengandalkan masukan terurut per key, itulah sebabnya langkah `sort` mutlak diperlukan di antara mapper dan reducer (dibuktikan pada L-6).

Mapper diuji pada tiga baris pertama, lalu seluruh pipeline dijalankan dengan pipe Linux. Perintah `sort` berperan sebagai **shuffle**, dan `LC_ALL=C` memastikan urutan berdasarkan byte, sama seperti pengurutan key di Hadoop.

In [ ]:
!head -n 3 masukan.tsv | python3 mapper.py
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m1.895s
user	0m2.047s
sys	0m0.094s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


**Penjelasan kode.** Baris pertama menguji mapper pada tiga baris pertama. Baris kedua menjalankan seluruh pipeline dengan pipe Linux: `cat masukan.tsv | mapper | sort | reducer`, dengan `sort` berperan sebagai **shuffle** dan `LC_ALL=C` memastikan urutan berdasarkan byte seperti pengurutan key di Hadoop. `time` mengukur lamanya, `wc -l` menghitung baris hasil, dan `sort -k2,2nr | head` menampilkan lima kata terbanyak.

**Interpretasi hasil.** Mapper mencetak satu baris `kata<TAB>1` untuk tiap kata (tiga judul pertama menghasilkan kata seperti `kemnaker`, `awasi`, `tka`, `di`, `meikarta`, ...). Hasil pipeline berisi **32.275 baris**, sama dengan jumlah kata unik pada H-9, dan lima kata teratas adalah `di` 27.390, `corona` 17.239, `ini` 10.124, `yang` 8.668, dan `dan` 6.785, persis sama dengan H-9. Waktu `real` bergantung pada komputer. Pesan `sort: write failed ... Broken pipe` yang mungkin muncul di akhir tidak berbahaya: `head` menutup pipe setelah menerima lima baris.

Terakhir, dibuktikan bahwa hasil pipe sama dengan hasil H-9.

In [ ]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


**Penjelasan kode.** Berkas `hasil_pipe.txt` dibaca kembali menjadi dictionary `kata -> jumlah`, lalu dibandingkan dengan `hasil_semua` dari H-9.

**Interpretasi hasil.** Hasilnya `True`: **keluaran Hadoop Streaming gaya pipe sama persis dengan hasil MapReduce Python pada H-9** untuk seluruh 32.275 kata. Artinya mapper dan reducer dapat diuji dulu dengan pipe Linux sebelum dijalankan di cluster, dan logikanya tidak bergantung pada pustaka pandas.

## H-13. Kata Teratas per Kategori, Ekspor ala Sqoop, dan Rangkuman

Pada langkah terakhir, key diganti menjadi pasangan `(kategori, kata)` dengan stopword dibuang di sisi mapper. Mapper memproses satu partisi dan combiner dipasang di dalam mapper. Hasilnya lalu diekspor ke database SQLite, seperti `sqoop export` yang memindahkan hasil Hadoop ke database relasional.

In [ ]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))
for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


**Penjelasan kode.** Key diganti menjadi pasangan `(kategori, kata)` dengan stopword dibuang di sisi mapper. `mapper_kategori()` memproses satu partisi dan memasang combiner (`Counter`) di dalamnya. Hasil semua mapper digabung, lima kata teratas tiap kategori dicetak, lalu hasilnya diekspor: ke Parquet (`siap_pakai/kata_per_kategori.parquet`) dan ke database SQLite `laporan.db` lewat `to_sql`, seperti `sqoop export` yang memindahkan hasil Hadoop ke database relasional.

**Interpretasi hasil.** Kata "corona" menjadi kata teratas pada **7 dari 9 kategori** (finance 2.470, health 2.590, hot 1.658, inet 845, news 7.539, sport 228, travel 1.023); pengecualiannya `food` (makanan 699, makan 539, foto 458) dan `oto` (mobil 1.205, corona hanya urutan kedua dengan 576). Kata khas tiap kategori terlihat jelas: `polisi`, `kasus`, `warga` pada news, `wisata` pada travel, `motogp` pada sport, `ponsel` pada inet. Kata `rp`, `ri`, dan `the` ikut muncul karena belum masuk stopword. Sebanyak **73.601 baris** (pasangan kategori-kata unik) diekspor ke database, dan tiga kata teratas kategori `health` adalah **corona (2.590), virus (1.040), dan pasien (471)**. Alur pipeline kini lengkap: portal, channel, pembersihan, partisi, MapReduce, hingga database laporan.

# L. Analisis Hasil

### L-1. Mengapa 2 proses tanpa combiner bisa lebih lambat daripada 1 proses? Apa yang berubah ketika combiner dipasang?

In [ ]:
import pickle

def waktu_median(fn, proses, n=3):
    w = []
    for _ in range(n):
        _, t_ = jalankan(fn, splits, proses); w.append(t_)
    return sorted(w)[1]

waktu_uji = {}
print(f"{'skenario':<28}{'waktu (s)':>10}")
for proses in (1, 2):
    for nama, fn in (("tanpa combiner", mapper_split), ("dengan combiner", mapper_combiner)):
        waktu_uji[(proses, nama)] = waktu_median(fn, proses)
        print(f"{proses} proses, {nama}".ljust(28), f"{waktu_uji[(proses, nama)]:>10.3f}")

# Biaya memindahkan hasil mapper antar-proses: serialisasi (pickle) lalu deserialisasi
def biaya_pindah(data):
    t0 = time.perf_counter()
    blob = [pickle.dumps(x) for x in data]
    _ = [pickle.loads(b) for b in blob]
    return sum(len(b) for b in blob) / 1e6, time.perf_counter() - t0

mb_tanpa, t_tanpa = biaya_pindah(hasil_map)
mb_dengan, t_dengan = biaya_pindah(h2)
print(f"\nPasangan yang dikirim : {sum(len(x) for x in hasil_map):,} (tanpa) vs {sum(len(x) for x in h2):,} (dengan combiner)")
print(f"Data antar-proses     : {mb_tanpa:.1f} MB vs {mb_dengan:.2f} MB (selisih {mb_tanpa/mb_dengan:.1f}x)")
print(f"Waktu serialisasi saja: {t_tanpa:.3f} s vs {t_dengan:.3f} s")
print("\n2 proses tanpa combiner lebih lambat dari 1 proses?             ", waktu_uji[(2, 'tanpa combiner')] > waktu_uji[(1, 'tanpa combiner')])
print("2 proses dengan combiner lebih cepat dari 2 proses tanpa combiner?", waktu_uji[(2, 'dengan combiner')] < waktu_uji[(2, 'tanpa combiner')])

skenario                     waktu (s)
1 proses, tanpa combiner          0.592
1 proses, dengan combiner         0.792
2 proses, tanpa combiner          2.608
2 proses, dengan combiner         0.872

Pasangan yang dikirim : 825,601 (tanpa) vs 74,009 (dengan combiner)
Data antar-proses     : 10.4 MB vs 0.87 MB (selisih 12.0x)
Waktu serialisasi saja: 0.785 s vs 0.031 s

2 proses tanpa combiner lebih lambat dari 1 proses?              True
2 proses dengan combiner lebih cepat dari 2 proses tanpa combiner? True


**Penjelasan kode.** Sel ini mengukur waktu median untuk empat skenario (1 atau 2 proses, tanpa atau dengan combiner), lalu menghitung biaya pemindahan hasil mapper antar-proses, yaitu jumlah pasangan yang dikirim, ukuran data setelah serialisasi, serta waktu serialisasi dan deserialisasi. Dua pernyataan perbandingan di akhir sel digunakan untuk memverifikasi apakah penambahan proses dan penggunaan combiner benar-benar memberikan keuntungan kinerja.

**Interpretasi hasil.** Pada pengujian ini, **1 proses tanpa combiner** membutuhkan waktu **0,592 detik**, sedangkan **2 proses tanpa combiner** membutuhkan **2,608 detik**, sehingga penggunaan 2 proses justru lebih lambat. Hal ini terjadi karena setiap proses harus mengirim hasilnya ke proses induk, menghasilkan **825.601 pasangan** atau sekitar **10,4 MB** data yang perlu diserialisasi. Waktu yang dihabiskan hanya untuk serialisasi mencapai **0,785 detik**.

Ketika **combiner** digunakan, jumlah pasangan yang dikirim turun menjadi **74.009** dan ukuran data yang diserialisasi menjadi sekitar **0,87 MB** (**12 kali lebih kecil**). Akibatnya, waktu eksekusi berkurang menjadi **0,792 detik** untuk 1 proses dan **0,872 detik** untuk 2 proses. Hasil ini menunjukkan bahwa pada pekerjaan dengan output mapper yang sangat besar, combiner lebih efektif dalam meningkatkan kinerja dibanding sekadar menambah jumlah proses karena mampu mengurangi biaya komunikasi dan serialisasi antar-proses secara signifikan.

### L-2. Mengapa menambah reducer dari 4 menjadi 64 tidak membuat beban reducer terberat mendekati bagian yang adil? Bagaimana combiner mengubah gambarnya?

In [ ]:
pasangan_combiner = [list(c.items()) for c in h2]
print(f"{'reducer':>8} {'terberat tanpa combiner':>25} {'terberat dengan combiner':>26} {'bagian adil':>13}")
for r in (4, 16, 64, 256):
    b1, b2 = beban(hasil_map, r), beban(pasangan_combiner, r)
    print(f"{r:>8} {max(b1)/sum(b1)*100:>24.2f}% {max(b2)/sum(b2)*100:>25.2f}% {100/r:>12.2f}%")

tot = sum(hasil_semua.values())
kata_top, n_top = collections.Counter(hasil_semua).most_common(1)[0]
print(f"\nKey terbesar '{kata_top}': {n_top:,} pasangan = {n_top/tot*100:.2f}% dari semua pasangan (tanpa combiner)")
n_mapper_di = sum(1 for c in h2 if kata_top in c)
tot_c = sum(len(h) for h in pasangan_combiner)
print(f"Dengan combiner, '{kata_top}' hanya {n_mapper_di} pasangan = {n_mapper_di/tot_c*100:.3f}% dari {tot_c:,} pasangan")

 reducer   terberat tanpa combiner   terberat dengan combiner   bagian adil
       4                    29.17%                     25.55%        25.00%
      16                     8.82%                      6.54%         6.25%
      64                     4.28%                      1.86%         1.56%
     256                     3.63%                      0.52%         0.39%

Key terbesar 'di': 27,390 pasangan = 3.32% dari semua pasangan (tanpa combiner)
Dengan combiner, 'di' hanya 9 pasangan = 0.012% dari 74,009 pasangan


**Penjelasan kode.** Sel ini menghitung beban reducer terberat untuk 4, 16, 64, dan 256 reducer, tanpa dan dengan combiner, lalu membandingkannya dengan bagian yang adil (100/R persen). Sel ini juga menunjukkan beban key terbesar (`di`) dan berapa pasangan yang tersisa untuk key itu setelah combiner.

**Interpretasi hasil (jawaban).** Menambah reducer dari 4 menjadi 64 tidak membuat reducer terberat mendekati bagian yang adil karena **satu key tidak bisa dipecah ke banyak reducer**: semua pasangan `di` (**27.390 pasangan = 3,32%** dari seluruh pasangan) harus ke reducer yang sama, sehingga reducer itu selalu menerima minimal 3,32%. Terberat tanpa combiner turun dari **29,17%** (4 reducer, adil 25,00%) menjadi **4,28%** (64 reducer, adil 1,56%, hampir tiga kali lipat), dan pada 256 reducer masih **3,63%** padahal bagian adilnya hanya 0,39% (sekitar sembilan kali lipat). **Combiner mengubah gambarnya** karena tiap kata hanya dikirim sekali per mapper: `di` tinggal **9 pasangan (0,012%)**, sehingga terberat menjadi 25,55% (4 reducer), 1,86% (64 reducer), dan 0,52% (256 reducer), jauh lebih dekat ke bagian yang adil. Skew berasal dari data, bukan dari jumlah mesin.

### L-3. Kata "rp", "ri", dan "the" muncul di antara kata teratas. Apakah sebaiknya ditambahkan ke daftar stopword?

In [ ]:
for kata in ("rp", "ri", "the"):
    per_kat = {kat: gabung.get((kat, kata), 0) for kat in sorted(per_kategori)}
    total_k = sum(per_kat.values())
    print(f"'{kata}': total {total_k:,} ->", {k: v for k, v in per_kat.items() if v})
    contoh_j = [t for t in bersih["judul"] if kata in tokenisasi(t)][:2]
    for t in contoh_j:
        print("     contoh:", t)

'rp': total 3,154 -> {'finance': 1385, 'food': 100, 'health': 26, 'hot': 315, 'inet': 210, 'news': 778, 'oto': 267, 'sport': 9, 'travel': 64}
     contoh: BRI Gelar Bazar Ragam Merek Sneaker, Ada Cashback hingga Rp 2 Juta
     contoh: Ada Anggaran Rp 72 M untuk Influencer, Dipakai Buat Apa Sih?
'ri': total 2,206 -> {'finance': 1125, 'food': 3, 'health': 196, 'hot': 6, 'inet': 19, 'news': 680, 'oto': 126, 'sport': 8, 'travel': 43}
     contoh: RI Bakal Punya Destinasi Wisata Sejarah Islam di Ancol
     contoh: CT Sebut Virus Baru Masuk RI, Namanya Jiwasraya
'the': total 678 -> {'finance': 8, 'food': 24, 'health': 4, 'hot': 528, 'inet': 16, 'news': 50, 'oto': 7, 'sport': 17, 'travel': 24}
     contoh: Erick Thohir Beberkan Rencana Bisnis 'The New Oil' Telkom
     contoh: Bahas Proyeksi Perikanan Budi Daya, Edhy Prabowo: Harus Out of The Box


**Penjelasan kode.** Untuk tiga kata (`rp`, `ri`, `the`), sel ini menghitung sebaran kemunculannya per kategori dari hasil H-13 dan menampilkan dua contoh judul yang memuatnya (pencocokan berbasis token agar `Rp1` tidak terlewat).

**Interpretasi hasil (jawaban).** `rp` muncul 3.154 kali (terbanyak di `finance`, 1.385 kali atau 44%) sebagai singkatan Rupiah, `ri` 2.206 kali (`finance` 1.125 kali atau 51%) sebagai singkatan Republik Indonesia, dan `the` 678 kali (`hot` 528 kali atau 78%) sebagai bagian judul berbahasa Inggris (misalnya "The New Oil"). Keputusannya bergantung pada **tujuan analisis**. Untuk pertanyaan redaksi (topik apa yang dibahas dan apakah berbeda antar-kategori), ketiganya adalah kata format atau kata fungsi yang tidak menyebut topik, dan karena sangat terkonsentrasi di satu kategori mereka justru menutupi kata bermakna pada kategori itu, jadi **sebaiknya ditambahkan ke stopword**. Jika tujuannya menganalisis pemberitaan ekonomi, `rp` justru penanda berita bernilai uang dan patut dipertahankan atau dihitung terpisah. Dampaknya terlihat pada N-1: setelah ditambahkan, lima kata teratas `finance` berganti menjadi corona, harga, jokowi, mau, ekonomi.

### L-4. Seluruh baris kategori inet punya url rusak. Apa dampaknya bila url dipakai sebagai kunci join? Mengapa barisnya tidak dibuang?

In [ ]:
inet_mentah = mentah[mentah["kategori"] == "inet"]
print("Nilai url pada kategori inet (mentah):", inet_mentah["url"].value_counts().to_dict())
inet_bersih = bersih[bersih["kategori"] == "inet"]
print("url kosong pada inet setelah dibersihkan:", int(inet_bersih["url"].isna().sum()), "dari", len(inet_bersih))
print("id kosong per kategori:", {k: int(v) for k, v in bersih.groupby("kategori")["id"].apply(lambda s: s.isna().sum()).items() if v})

# (a) Data mentah di-join dengan dirinya sendiri lewat url: jumlah baris hasil = jumlah kuadrat kemunculan tiap kunci
n_kunci = mentah["url"].value_counts()
print(f"\n(a) Mentah, join lewat url   : {int((n_kunci ** 2).sum()):,} baris, bukan {len(mentah):,}")

# (b) Data bersih: url kosong (NaN). Pandas menganggap NaN sama dengan NaN saat merge
n_kunci_b = bersih["url"].value_counts(dropna=False)
baris_b = int((n_kunci_b ** 2).sum())
print(f"(b) Bersih, join lewat url   : {baris_b:,} baris, bukan {len(bersih):,} (ledakan {baris_b/len(bersih):.1f}x)")

kecil = inet_bersih.head(100)[["judul", "url"]]
print(f"    Bukti kecil: 100 baris inet di-join lewat url -> {len(kecil.merge(kecil, on='url')):,} baris (seharusnya 100)")

# (c) SQL (DuckDB) berperilaku lain: NULL tidak pernah sama dengan NULL, sehingga baris inet hilang dari hasil join
con.register("kecil_df", kecil)
print("(c) DuckDB inner join lewat url untuk 100 baris yang sama ->", con.sql("SELECT COUNT(*) FROM kecil_df a JOIN kecil_df b ON a.url = b.url").fetchall()[0][0], "baris")

# Seandainya baris dengan url rusak dibuang
n_corona_inet = con.sql("SELECT COUNT(*) FROM berita WHERE kategori='inet' AND lower(judul) LIKE '%corona%'").fetchall()[0][0]
print(f"\nBila baris url rusak dibuang: kategori tersisa {bersih.loc[bersih['url'].notna(), 'kategori'].nunique()} dari {bersih['kategori'].nunique()}, "
      f"baris hilang {len(inet_bersih):,} ({len(inet_bersih)/len(bersih)*100:.2f}% data bersih), termasuk {n_corona_inet:,} judul 'corona'")

Nilai url pada kategori inet (mentah): {'#': 5640}
url kosong pada inet setelah dibersihkan: 5577 dari 5577
id kosong per kategori: {'inet': 5577}

(a) Mentah, join lewat url   : 31,894,977 baris, bukan 91,017
(b) Bersih, join lewat url   : 31,188,170 baris, bukan 90,818 (ledakan 343.4x)
    Bukti kecil: 100 baris inet di-join lewat url -> 10,000 baris (seharusnya 100)
(c) DuckDB inner join lewat url untuk 100 baris yang sama -> 0 baris

Bila baris url rusak dibuang: kategori tersisa 8 dari 9, baris hilang 5,577 (6.14% data bersih), termasuk 847 judul 'corona'


**Penjelasan kode.** Sel ini memeriksa nilai `url` kategori `inet` pada data mentah dan bersih, lalu menghitung akibatnya bila `url` dipakai sebagai kunci join: (a) data mentah di-join dengan dirinya sendiri (jumlah baris hasil join sama dengan jumlah kuadrat kemunculan tiap nilai kunci), (b) data bersih yang `url`-nya kosong (pandas menganggap `NaN` sama dengan `NaN` saat merge), dibuktikan dengan merge sungguhan pada 100 baris, dan (c) perilaku SQL pada DuckDB yang menganggap `NULL` tidak pernah sama dengan `NULL`. Dihitung juga akibat membuang baris tersebut.

**Interpretasi hasil (jawaban).** Seluruh **5.640** baris `inet` pada data mentah memiliki `url` yang sama, yaitu `#`. Setelah dibersihkan, `url` dan `id` kosong untuk **semua 5.577** baris `inet` dan tidak ada kategori lain yang `id`-nya kosong. **Dampak bila `url` dipakai sebagai kunci join:** (a) pada data mentah, nilai `#` yang sama pada ribuan baris menyebabkan **ledakan baris**: join dengan dirinya sendiri menghasilkan sekitar **31,9 juta baris**, padahal data hanya 91.017 baris; (b) pada data bersih, pandas mencocokkan `NaN` dengan `NaN`, sehingga join menghasilkan **31.188.170 baris** (ledakan **343,4 kali**), dan 100 baris `inet` saja sudah menjadi **10.000 baris**; (c) pada SQL (DuckDB) hasilnya sebaliknya, **0 baris**, karena `NULL` tidak pernah sama dengan `NULL`, jadi baris `inet` hilang diam-diam. Jadi kunci yang rusak menimbulkan kesalahan yang berbeda bergantung pada mesinnya, dan keduanya salah. **Mengapa barisnya tidak dibuang:** `inet` adalah **6,14%** data bersih, satu kategori utuh (teknologi) yang akan hilang sehingga tersisa 8 dari 9 kategori, termasuk **847 judul** tentang corona. Judulnya masih berguna untuk analisis teks, sehingga yang diperbaiki hanya kolom `url` (dikosongkan), dan keputusan ini dicatat sebagai keputusan analisis.

### L-5. Mengapa partisi kategori dipilih dibanding kategori + bulan untuk MapReduce di H-9?

In [ ]:
r1 = bersih.groupby("kategori").size()
r2 = bersih.groupby(["kategori", "bulan"]).size()
print(f"Partisi kategori        : {len(r1):>2} split | judul per split: min {r1.min():,}, median {int(r1.median()):,}, maks {r1.max():,}")
print(f"Partisi kategori+bulan  : {len(r2):>2} split | judul per split: min {r2.min():,}, median {int(r2.median()):,}, maks {r2.max():,}")

simpan_terpartisi(bersih, ["kategori", "bulan"], "uji_partisi2")
splits2 = sorted(Path("uji_partisi2").rglob("*.parquet"))

def ukuran_kb(files):
    u = [p.stat().st_size / 1024 for p in files]
    return sum(u) / len(u), min(u), max(u)
for nama, fs in (("kategori", splits), ("kategori+bulan", splits2)):
    rata, kecil_, besar = ukuran_kb(fs)
    print(f"Ukuran split {nama:<15}: rata-rata {rata:7.1f} KB | terkecil {kecil_:7.1f} KB | terbesar {besar:7.1f} KB")

w1, w2 = [], []
for _ in range(3):
    w1.append(jalankan(mapper_split, splits, 1)[1])
    w2.append(jalankan(mapper_split, splits2, 1)[1])
print(f"\nWaktu mapper (1 proses, median 3x): {len(splits)} split = {sorted(w1)[1]:.3f} s | {len(splits2)} split = {sorted(w2)[1]:.3f} s")

hasil_p2 = collections.Counter()
for c in jalankan(mapper_combiner, splits2, 1)[0]:
    hasil_p2.update(c)
print("Word count dengan partisi kategori+bulan sama dengan H-9:", dict(hasil_p2) == hasil_semua)
shutil.rmtree("uji_partisi2")

Partisi kategori        :  9 split | judul per split: min 2,433, median 5,577, maks 32,321
Partisi kategori+bulan  : 54 split | judul per split: min 190, median 1,043, maks 6,633
Ukuran split kategori       : rata-rata   907.6 KB | terkecil   208.3 KB | terbesar  3148.7 KB
Ukuran split kategori+bulan : rata-rata   152.7 KB | terkecil    21.4 KB | terbesar   633.3 KB

Waktu mapper (1 proses, median 3x): 9 split = 0.840 s | 54 split = 1.173 s
Word count dengan partisi kategori+bulan sama dengan H-9: True


**Penjelasan kode.** Sel ini membandingkan sebaran judul dan ukuran berkas per split untuk dua skema partisi, menulis sementara versi `kategori + bulan` ke `uji_partisi2/`, mengukur waktu mapper pada kedua skema (satu proses, median tiga kali), memeriksa bahwa hasil word count tetap identik, lalu menghapus folder sementara.

**Interpretasi hasil (jawaban).** Partisi `kategori` menghasilkan **9 split** (min 2.433, median 5.577, maks 32.321 judul; ukuran berkas rata-rata sekitar 0,9 MB, dari sekitar 0,2 sampai 3,2 MB), sedangkan `kategori + bulan` menghasilkan **54 split** (min 190, median 1.043, maks 6.633 judul; rata-rata sekitar 0,15 MB, dari sekitar 0,02 sampai 0,65 MB). Jumlah berkas parquet sama dengan jumlah input split sehingga jumlah mapper naik dari **9 menjadi 54**. Total ukuran data hampir sama (sekitar 8,4 MB), jadi tiap berkas menjadi kecil: inilah **small file problem**, karena setiap mapper punya biaya awal (membuka berkas, membaca metadata Parquet, menjadwalkan tugas) yang tidak sebanding dengan kerja yang sedikit. Waktu fase map pada 54 split sedikit lebih lama daripada 9 split (lihat keluaran; pada sesi Colab acuan sekitar 1,25 kali), padahal hasil word count **identik** (`True`), jadi partisi yang lebih halus tidak memberi keuntungan. Partisi `kategori` juga selaras dengan pertanyaan analisis (perbedaan topik antar-kategori) dan dengan key `(kategori, kata)` pada H-13, tempat satu mapper memproses satu kategori utuh.

### L-6. Mengapa hasil harus diurutkan sebelum masuk ke reducer.py? Apa yang terjadi bila sort dihilangkan?

In [ ]:
import subprocess

def jalankan_pipe(pakai_sort):
    perintah = "cat masukan.tsv | python3 mapper.py | " + ("LC_ALL=C sort | " if pakai_sort else "") + "python3 reducer.py"
    out = subprocess.run(perintah, shell=True, capture_output=True, text=True, executable="/bin/bash").stdout
    return [b.split("\t") for b in out.strip().split("\n")]

dengan_sort, tanpa_sort = jalankan_pipe(True), jalankan_pipe(False)
print(f"Dengan sort : {len(dengan_sort):>8,} baris keluaran | key unik: {len({k for k, _ in dengan_sort}):,}")
print(f"Tanpa sort  : {len(tanpa_sort):>8,} baris keluaran | key unik: {len({k for k, _ in tanpa_sort}):,}")

kemunculan = collections.Counter(k for k, _ in tanpa_sort)
print("Kata yang muncul lebih dari satu kali pada keluaran tanpa sort:", f"{sum(1 for v in kemunculan.values() if v > 1):,}")
for kata_uji in ("di", "corona"):
    tanpa_k = [int(n) for k, n in tanpa_sort if k == kata_uji]
    asli = [int(n) for k, n in dengan_sort if k == kata_uji]
    print(f"'{kata_uji}': dengan sort {asli} | tanpa sort {len(tanpa_k):,} baris terpisah, lima pertama {tanpa_k[:5]}, jumlahnya {sum(tanpa_k):,}")

print("\nHasil tanpa sort sama dengan H-9 bila dibaca sebagai kamus:", {k: int(n) for k, n in tanpa_sort} == hasil_semua)
print("Jumlah seluruh hitungan sama:", sum(int(n) for _, n in dengan_sort) == sum(int(n) for _, n in tanpa_sort) == total_pasangan)

Dengan sort :   32,275 baris keluaran | key unik: 32,275
Tanpa sort  :  821,949 baris keluaran | key unik: 32,275
Kata yang muncul lebih dari satu kali pada keluaran tanpa sort: 21,214
'di': dengan sort [27390] | tanpa sort 27,387 baris terpisah, lima pertama [1, 1, 1, 1, 1], jumlahnya 27,390
'corona': dengan sort [17239] | tanpa sort 17,210 baris terpisah, lima pertama [1, 1, 1, 1, 1], jumlahnya 17,239

Hasil tanpa sort sama dengan H-9 bila dibaca sebagai kamus: False
Jumlah seluruh hitungan sama: True


**Penjelasan kode.** Pipeline Hadoop Streaming dijalankan dua kali lewat `subprocess`: dengan `sort` (benar) dan tanpa `sort` (mapper langsung ke reducer). Jumlah baris keluaran, jumlah key unik, kata yang muncul lebih dari satu kali, dan hasil untuk `di` serta `corona` dibandingkan, lalu dicek apakah keluaran tanpa `sort` sama dengan hasil H-9 dan apakah jumlah seluruh hitungan tetap sama.

**Interpretasi hasil (jawaban).** `reducer.py` hanya menjumlahkan pasangan yang **berurutan** dengan key sama, dan mencetak total ketika key berganti, sehingga ia mengandalkan masukan yang sudah dikelompokkan per key oleh `sort` (shuffle). Dengan `sort`, keluaran berisi **32.275 baris**, satu baris per key (`di` = 27.390, `corona` = 17.239). **Tanpa `sort`**, keluaran membengkak menjadi **821.949 baris** padahal key uniknya tetap 32.275: **21.214 kata** muncul lebih dari satu kali sebagai total parsial. Kata `di` muncul sebagai **27.387 baris terpisah** (kebanyakan bernilai 1) dan `corona` sebagai **17.210 baris**, yang baru jika dijumlahkan kembali menjadi 27.390 dan 17.239. Jumlah seluruh hitungan tetap sama (825.601), tetapi hasil dibaca sebagai kamus **tidak sama** dengan H-9 (`False`), karena tidak ada satu baris pun yang memuat total kata sebenarnya. Karena itu shuffle atau sort wajib berjalan di antara mapper dan reducer.

### L-7. Mengapa H-4 memakai sitemap berita dan bukan membuka halaman setiap artikel?

In [ ]:
html_lama = ('<article class="item"><h2 class="judul"><a href="u">Judul A</a></h2>'
             '<span class="kategori">news</span><time datetime="2020-01-01">01/01/2020</time></article>')
html_baru = html_lama.replace('class="item"', 'class="kartu"')   # tampilan situs berubah
print("parse_halaman pada HTML lama :", len(parse_halaman(html_lama)), "judul")
print("parse_halaman pada HTML baru :", len(parse_halaman(html_baru)), "judul (gagal diam-diam)")
print("parse_sitemap_news pada XML  :", len(parse_sitemap_news(CONTOH_XML)), "judul (tidak bergantung tampilan)")

print(f"\nRequest ke server : {len(SITEMAP_DIPILIH)} sitemap vs {BATAS_TOTAL} halaman artikel untuk {BATAS_TOTAL} judul")
print(f"Jeda minimum      : {len(SITEMAP_DIPILIH) * DELAY:.0f} detik vs {BATAS_TOTAL * DELAY:.0f} detik (DELAY = {DELAY:.0f} detik)")
print("Isi tiap entri sitemap :", sorted(parse_sitemap_news(CONTOH_XML)[0].keys()))

parse_halaman pada HTML lama : 1 judul
parse_halaman pada HTML baru : 0 judul (gagal diam-diam)
parse_sitemap_news pada XML  : 3 judul (tidak bergantung tampilan)

Request ke server : 3 sitemap vs 20 halaman artikel untuk 20 judul
Jeda minimum      : 6 detik vs 40 detik (DELAY = 2 detik)
Isi tiap entri sitemap : ['id', 'judul', 'kata_kunci', 'kategori', 'tanggal', 'url']


**Penjelasan kode.** Sel ini membandingkan ketahanan dua pendekatan: `parse_halaman` pada HTML lama dan HTML yang tampilannya berubah (nama class diganti), dibandingkan `parse_sitemap_news` yang membaca XML. Sel ini juga menghitung jumlah request dan jeda minimum untuk 20 judul dengan kedua cara.

**Interpretasi hasil (jawaban).** Bila tampilan situs berubah, `parse_halaman` mengembalikan **0 judul tanpa pesan galat** (gagal diam-diam), sedangkan `parse_sitemap_news` tetap mendapat **3 judul** karena XML sitemap adalah format data yang terdokumentasi dan tidak ikut berubah. Satu entri sitemap sudah memuat `judul`, `url`, `kategori`, `id`, `tanggal`, dan `kata_kunci`, jadi cukup **3 request sitemap** (jeda minimum **6 detik**) dibandingkan **20 request halaman artikel** (**40 detik**) untuk 20 judul. **Untung bagi server detik:** beban jauh lebih ringan dan hanya berkas XML kecil yang diminta. **Untung bagi kode kita:** lebih ringan, lebih tahan perubahan tampilan, dan sudah memuat tanggal serta kata kunci tanpa membuka artikel. **Ruginya:** hanya mendapat metadata terbaru (satu sitemap berita berisi sekitar 200 entri), bukan isi artikel atau arsip lama, bergantung pada tersedianya sitemap, dan tanda zona waktunya tampak keliru sehingga hanya tanggal yang dipakai.

# M. Studi Kasus: Apa yang Dibahas Portal Berita pada Paruh Pertama 2020?

Pimpinan redaksi meminta ringkasan satu halaman berdasarkan data judul. Sel berikut menyiapkan angka pendukung dari H-7 dan H-13: proporsi judul "corona" per bulan dan per kategori, serta kata teratas tiap kategori di luar "corona".

In [ ]:
corona_bulan = con.sql("""
    SELECT bulan, COUNT(*) AS judul,
           COUNT(*) FILTER (WHERE lower(judul) LIKE '%corona%') AS corona
    FROM berita GROUP BY bulan ORDER BY bulan""").df()
corona_bulan["persen"] = (corona_bulan["corona"] / corona_bulan["judul"] * 100).round(1)
print(corona_bulan.to_string(index=False))

corona_kat = con.sql("""
    SELECT kategori, COUNT(*) AS judul,
           COUNT(*) FILTER (WHERE lower(judul) LIKE '%corona%') AS corona
    FROM berita GROUP BY kategori""").df()
corona_kat["persen"] = (corona_kat["corona"] / corona_kat["judul"] * 100).round(1)
print()
print(corona_kat.sort_values("persen", ascending=False).to_string(index=False))

print("\nKata teratas per bulan (tanpa stopword, rp/ri/the diabaikan):")
for b in ("2020-01", "2020-05"):
    cb = collections.Counter(k for t in bersih.loc[bersih["bulan"] == b, "judul"] for k in tokenisasi(t)
                             if k not in STOPWORD and k not in {"rp", "ri", "the"})
    print(f"  {b}:", ", ".join(f"{k} ({n:,})" for k, n in cb.most_common(6)))

print("\nKata teratas selain 'corona' per kategori:")
for kat in sorted(per_kategori):
    lain = [f"{k} ({n:,})" for n, k in sorted(per_kategori[kat], reverse=True) if k != "corona"][:3]
    print(f"  {kat:<8}", ", ".join(lain))

  bulan  judul  corona  persen
2020-01  16258     385     2.4
2020-02  16135    1077     6.7
2020-03  18103    6994    38.6
2020-04  16636    4913    29.5
2020-05  16352    2934    17.9
2020-06   7334     943    12.9

kategori  judul  corona  persen
  health   4914    2591    52.7
    news  32321    7532    23.3
 finance  14141    2473    17.5
  travel   6453    1025    15.9
    inet   5577     847    15.2
     oto   4375     577    13.2
     hot  16290    1663    10.2
   sport   2433     228     9.4
    food   4314     310     7.2

Kata teratas per bulan (tanpa stopword, rp/ri/the diabaikan):
  2020-01: banjir (1,220), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428)
  2020-05: corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562)

Kata teratas selain 'corona' per kategori:
  finance  rp (1,385), ri (1,125), harga (701)
  food     makanan (699), makan (539), foto (458)
  health   virus (1,040), pasien (471), indonesia (427)
  hot      an

**Penjelasan kode.** Dengan DuckDB, sel ini menghitung jumlah judul dan judul yang memuat "corona" per bulan dan per kategori beserta persentasenya, lalu mencetak kata teratas pada Januari dan Mei (kata `rp`, `ri`, dan `the` diabaikan) serta tiga kata teratas selain "corona" untuk tiap kategori (dari hasil H-13).

**Interpretasi hasil.** Judul "corona" memuncak pada **Maret 2020: 6.994 judul (38,6%** dari seluruh judul bulan itu), naik dari 385 judul (2,4%) pada Januari, lalu turun menjadi 4.913 (29,5%) pada April, 2.934 (17,9%) pada Mei, dan 943 (12,9%) pada 13 hari pertama Juni. Totalnya **17.246 judul (19,0% dari 90.818)**. Dari sisi kategori, **health paling intens (52,7%)**, disusul news (23,3%), finance (17,5%), travel (15,9%), inet (15,2%), oto (13,2%), hot (10,2%), sport (9,4%), dan **food paling rendah (7,2%)**. Secara jumlah judul, news terbanyak (7.532). Pergeseran topik terlihat dari kata teratas: Januari didominasi banjir (1.220), polisi, dan Jokowi, sedangkan Mei didominasi corona (2.935), kasus, positif, pandemi, dan lebaran. Kata khas kategori juga berbeda: food (makanan, makan, foto), oto (mobil, motor), dan sport (motogp, masters).

In [ ]:
from IPython.display import display, Markdown

RINGKASAN = (
    'Selama 1 Januari sampai 13 Juni 2020, pemberitaan portal ini didominasi pandemi. Judul yang memuat '
    'kata "corona" memuncak pada Maret 2020 dengan 6.994 judul (38,6% dari seluruh judul bulan itu), naik '
    'tajam dari 385 judul (2,4%) pada Januari, lalu menurun menjadi 4.913 pada April dan 2.934 pada Mei. '
    'Pergeseran topik terlihat jelas: Januari didominasi banjir, polisi, dan Jokowi, sedangkan Mei '
    'didominasi corona, kasus, positif, pandemi, dan PSBB. Dalam jumlah judul, kategori news paling '
    'banyak membahas corona (7.532 judul), tetapi secara proporsi kategori health paling intens, karena '
    '52,7% judulnya memuat kata itu. Kategori yang bahasannya paling berbeda adalah food: hanya 7,2% '
    'judulnya menyebut corona dan kata teratasnya makanan, makan, serta foto; kategori oto juga berbeda '
    'karena dipimpin kata mobil dan motor. Keterbatasan yang perlu diketahui pimpinan: data hanya berisi '
    'judul, bukan isi artikel, sehingga frekuensi kata belum menunjukkan sudut pandang berita; Juni hanya '
    'mencakup 13 hari sehingga tidak sebanding dengan bulan lain; dan dataset dikumpulkan pihak ketiga '
    'tanpa lisensi, sehingga hanya layak untuk pembelajaran.'
)

n_kata = len(RINGKASAN.split())
print("Jumlah kata ringkasan:", n_kata, "| dalam rentang 150-200:", 150 <= n_kata <= 200)
print("Memuat (a) bulan puncak          :", "Maret 2020" in RINGKASAN and "6.994" in RINGKASAN)
print("Memuat (b) kategori terbanyak    :", "news" in RINGKASAN and "health" in RINGKASAN)
print("Memuat (b) kategori paling beda  :", "food" in RINGKASAN)
print("Memuat (c) keterbatasan data     :", "hanya berisi judul" in RINGKASAN and "13 hari" in RINGKASAN)
print()
display(Markdown(RINGKASAN))

Jumlah kata ringkasan: 163 | dalam rentang 150-200: True
Memuat (a) bulan puncak          : True
Memuat (b) kategori terbanyak    : True
Memuat (b) kategori paling beda  : True
Memuat (c) keterbatasan data     : True



Selama 1 Januari sampai 13 Juni 2020, pemberitaan portal ini didominasi pandemi. Judul yang memuat kata "corona" memuncak pada Maret 2020 dengan 6.994 judul (38,6% dari seluruh judul bulan itu), naik tajam dari 385 judul (2,4%) pada Januari, lalu menurun menjadi 4.913 pada April dan 2.934 pada Mei. Pergeseran topik terlihat jelas: Januari didominasi banjir, polisi, dan Jokowi, sedangkan Mei didominasi corona, kasus, positif, pandemi, dan PSBB. Dalam jumlah judul, kategori news paling banyak membahas corona (7.532 judul), tetapi secara proporsi kategori health paling intens, karena 52,7% judulnya memuat kata itu. Kategori yang bahasannya paling berbeda adalah food: hanya 7,2% judulnya menyebut corona dan kata teratasnya makanan, makan, serta foto; kategori oto juga berbeda karena dipimpin kata mobil dan motor. Keterbatasan yang perlu diketahui pimpinan: data hanya berisi judul, bukan isi artikel, sehingga frekuensi kata belum menunjukkan sudut pandang berita; Juni hanya mencakup 13 hari sehingga tidak sebanding dengan bulan lain; dan dataset dikumpulkan pihak ketiga tanpa lisensi, sehingga hanya layak untuk pembelajaran.

**Penjelasan kode.** Ringkasan 150 sampai 200 kata untuk pimpinan redaksi disimpan di variabel `RINGKASAN`. Kode memeriksa secara otomatis jumlah katanya dan keberadaan tiga unsur : (a) bulan puncak judul "corona", (b) kategori yang paling banyak membahas dan yang bahasannya paling berbeda, dan (c) satu keterbatasan data. Teks lalu ditampilkan sebagai Markdown.

**Interpretasi hasil.** Ringkasan terdiri atas **163 kata** (dalam rentang 150 sampai 200) dan keempat pemeriksaan unsur bernilai `True`. Isi ringkasan:

> Selama 1 Januari sampai 13 Juni 2020, pemberitaan portal ini didominasi pandemi. Judul yang memuat kata "corona" memuncak pada Maret 2020 dengan 6.994 judul (38,6% dari seluruh judul bulan itu), naik tajam dari 385 judul (2,4%) pada Januari, lalu menurun menjadi 4.913 pada April dan 2.934 pada Mei. Pergeseran topik terlihat jelas: Januari didominasi banjir, polisi, dan Jokowi, sedangkan Mei didominasi corona, kasus, positif, pandemi, dan PSBB. Dalam jumlah judul, kategori news paling banyak membahas corona (7.532 judul), tetapi secara proporsi kategori health paling intens, karena 52,7% judulnya memuat kata itu. Kategori yang bahasannya paling berbeda adalah food: hanya 7,2% judulnya menyebut corona dan kata teratasnya makanan, makan, serta foto; kategori oto juga berbeda karena dipimpin kata mobil dan motor. Keterbatasan yang perlu diketahui pimpinan: data hanya berisi judul, bukan isi artikel, sehingga frekuensi kata belum menunjukkan sudut pandang berita; Juni hanya mencakup 13 hari sehingga tidak sebanding dengan bulan lain; dan dataset dikumpulkan pihak ketiga tanpa lisensi, sehingga hanya layak untuk pembelajaran.

Seluruh angka dalam ringkasan berasal dari sel sebelumnya: bulan dan proporsi dari query DuckDB (H-7 dan sel di atas), kata teratas dari H-13 dan latihan N-2.

# N. Latihan

### N-1. Stopword domain

In [ ]:
STOPWORD_BARU = STOPWORD | {"rp", "ri", "the", "foto", "galeri"}

def mapper_kategori_sw(path, stop):
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in stop:
                c[(kategori, kata)] += 1
    return c

gabung_baru = collections.Counter()
for p in splits:
    gabung_baru.update(mapper_kategori_sw(p, STOPWORD_BARU))

def top5_per_kategori(g):
    out = collections.defaultdict(list)
    for (kat, kata), n in g.items():
        out[kat].append((n, kata))
    return {kat: [k for n, k in sorted(v, reverse=True)[:5]] for kat, v in out.items()}

lama, baru = top5_per_kategori(gabung), top5_per_kategori(gabung_baru)
print("Pasangan (kategori, kata) sebelum:", f"{len(gabung):,}", "| sesudah:", f"{len(gabung_baru):,}")
print("Kata yang ditambahkan:", sorted(STOPWORD_BARU - STOPWORD))
for kat in sorted(lama):
    if lama[kat] != baru[kat]:
        print(f"{kat:<8} sebelum: {', '.join(lama[kat])}")
        print(f"{'':<8} sesudah: {', '.join(baru[kat])}")
    else:
        print(f"{kat:<8} tidak berubah")

Pasangan (kategori, kata) sebelum: 73,601 | sesudah: 73,562
Kata yang ditambahkan: ['foto', 'galeri', 'ri', 'rp', 'the']
finance  sebelum: corona, rp, ri, harga, jokowi
         sesudah: corona, harga, jokowi, mau, ekonomi
food     sebelum: makanan, makan, foto, galeri, enak
         sesudah: makanan, makan, enak, bikin, rumah
health   tidak berubah
hot      sebelum: corona, anak, rumah, meninggal, the
         sesudah: corona, anak, rumah, meninggal, film
inet     tidak berubah
news     tidak berubah
oto      sebelum: mobil, corona, motor, rp, indonesia
         sesudah: mobil, corona, motor, indonesia, honda
sport    tidak berubah
travel   sebelum: corona, wisata, foto, dunia, turis
         sesudah: corona, wisata, dunia, turis, indonesia


**Penjelasan kode.** Lima kata ditambahkan ke stopword (`rp`, `ri`, `the`, `foto`, `galeri`) dalam himpunan baru `STOPWORD_BARU`, lalu H-13 dijalankan ulang dengan mapper yang menerima daftar stopword sebagai parameter (variabel dan berkas hasil H-13 asli tidak ditimpa). Lima kata teratas tiap kategori dibandingkan sebelum dan sesudah, begitu pula jumlah pasangan `(kategori, kata)`.

**Interpretasi hasil.** Kata yang ditambahkan: `foto`, `galeri`, `ri`, `rp`, `the`. Jumlah pasangan `(kategori, kata)` turun dari **73.601 menjadi 73.562** (39 pasangan karena kelima kata itu muncul di beberapa kategori). Lima kategori berubah: **finance** (corona, rp, ri, harga, jokowi menjadi corona, harga, jokowi, mau, ekonomi), **food** (makanan, makan, foto, galeri, enak menjadi makanan, makan, enak, bikin, rumah), **hot** (the keluar dan film masuk), **oto** (rp keluar dan honda masuk), dan **travel** (foto keluar dan indonesia masuk). Empat kategori tidak berubah: health, inet, news, dan sport, karena kelima kata tersebut tidak masuk lima besar mereka. Kata yang tadinya hanya format atau kata fungsi tergantikan oleh kata yang lebih informatif tentang topik, sehingga daftar stopword domain meningkatkan mutu hasil.

### N-2. Per bulan: key (bulan, kata)

In [ ]:
def mapper_bulan(path):
    df_ = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(df_["judul"], df_["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
for p in splits:
    gabung_bulan.update(mapper_bulan(p))

total_token = collections.Counter()
for (b, kata), n in gabung_bulan.items():
    total_token[b] += n

def top_bulan(b, k=10):
    return [(kata, n) for (bb, kata), n in gabung_bulan.most_common() if bb == b][:k]

jan, mei = top_bulan("2020-01"), top_bulan("2020-05")
print("Januari:", ", ".join(f"{k} ({n:,})" for k, n in jan))
print("Mei    :", ", ".join(f"{k} ({n:,})" for k, n in mei))
print(f"Total token (tanpa stopword): Januari {total_token['2020-01']:,} | Mei {total_token['2020-05']:,}")

# Porsi kata per 10.000 token (adil karena total token tiap bulan berbeda); hanya kata dengan >= 100 kemunculan pada dua bulan itu
def porsi(b, kata):
    return gabung_bulan.get((b, kata), 0) / total_token[b] * 10000
kandidat = {k for (b, k) in gabung_bulan if b in ("2020-01", "2020-05") and gabung_bulan.get(("2020-01", k), 0) + gabung_bulan.get(("2020-05", k), 0) >= 100}
selisih = sorted(((porsi("2020-05", k) - porsi("2020-01", k), k) for k in kandidat), reverse=True)
print("\nKata yang paling naik porsinya Januari -> Mei (per 10.000 token):")
for d, k in selisih[:8]:
    print(f"  {k:<12} {porsi('2020-01', k):6.1f} -> {porsi('2020-05', k):6.1f}  ({d:+.1f})")
print("Kata yang paling turun porsinya Januari -> Mei:")
for d, k in selisih[::-1][:8]:
    print(f"  {k:<12} {porsi('2020-01', k):6.1f} -> {porsi('2020-05', k):6.1f}  ({d:+.1f})")

Januari: banjir (1,220), rp (616), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428), minta (427), kpk (415), indonesia (410)
Mei    : corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), rp (558), covid (522), psbb (522), polisi (517)
Total token (tanpa stopword): Januari 124,779 | Mei 128,973

Kata yang paling naik porsinya Januari -> Mei (per 10.000 token):
  corona         30.9 ->  227.6  (+196.7)
  positif         1.6 ->   48.7  (+47.1)
  pandemi         0.0 ->   46.2  (+46.2)
  lebaran         0.7 ->   43.6  (+42.9)
  psbb            0.0 ->   40.5  (+40.5)
  covid           0.0 ->   40.5  (+40.5)
  normal          1.9 ->   39.1  (+37.2)
  new             1.1 ->   37.4  (+36.3)
Kata yang paling turun porsinya Januari -> Mei:
  banjir         97.8 ->    4.6  (-93.2)
  jiwasraya      26.8 ->    1.0  (-25.8)
  kpk            33.3 ->    7.5  (-25.7)
  lina           20.5 ->    0.2  (-20.4)
  natuna         18.9 ->    0.3  (-18.6)
  korb

**Penjelasan kode.** Key diubah menjadi `(bulan, kata)`. `mapper_bulan()` membaca kolom `judul` dan `bulan` tiap partisi, membuang stopword, dan menghitung pasangan `(bulan, kata)` dengan combiner `Counter`. Sepuluh kata teratas Januari dan Mei dibandingkan. Karena total token tiap bulan berbeda, perubahan diukur lewat **porsi per 10.000 token** (hanya kata dengan sedikitnya 100 kemunculan pada kedua bulan).

**Interpretasi hasil.** Sepuluh kata teratas **Januari**: banjir (1.220), rp (616), polisi (554), jokowi (516), tahun, baru, mobil, minta, kpk, indonesia. **Mei**: corona (2.935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), rp (558), covid (522), psbb (522), polisi (517). Total token (tanpa stopword) Januari 124.779 dan Mei 128.973, jadi keduanya sebanding. **Kata yang paling naik porsinya:** corona 30,9 menjadi 227,6 (+196,7), positif 1,6 menjadi 48,7, pandemi 0,0 menjadi 46,2, lebaran 0,7 menjadi 43,6, psbb dan covid 0,0 menjadi 40,5, normal 1,9 menjadi 39,1, dan new 1,1 menjadi 37,4 ("new normal"). **Kata yang paling turun:** banjir 97,8 menjadi 4,6 (-93,2), jiwasraya 26,8 menjadi 1,0, kpk 33,3 menjadi 7,5, lina 20,5 menjadi 0,2, natuna 18,9 menjadi 0,3, korban 25,7 menjadi 9,4, china 30,9 menjadi 15,0, dan anies 26,4 menjadi 12,6. Pergeseran dari banjir, Jiwasraya, Natuna, dan KPK pada Januari ke pandemi, PSBB, dan lebaran pada Mei sesuai garis waktu peristiwa awal 2020. `rp` dan `polisi` konsisten di kedua bulan.

### N-3. Bigram

In [ ]:
def mapper_bigram(path):
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    out = []
    for t in judul:
        tok = tokenisasi(t)
        out += [(a + " " + b, 1) for a, b in zip(tok, tok[1:])]
    return out

hasil_bg = [mapper_bigram(p) for p in splits]
total_bg = sum(len(h) for h in hasil_bg)

setelah_bg = []
for h in hasil_bg:
    c = collections.Counter()
    for k, n in h:
        c[k] += n
    setelah_bg.append(c)
total_bg_c = sum(len(c) for c in setelah_bg)

print(f"Pasangan bigram tanpa combiner : {total_bg:,}")
print(f"Pasangan bigram dengan combiner: {total_bg_c:,} ({total_bg/total_bg_c:.1f}x lebih sedikit)")
print(f"Selisih dengan pasangan kata H-9 ({total_pasangan:,}): {total_pasangan - total_bg:,} = jumlah judul ({len(bersih):,})")

gabung_bg = collections.Counter()
for c in setelah_bg:
    gabung_bg.update(c)
print(f"Bigram unik: {len(gabung_bg):,}")
print("\n5 bigram teratas:", ", ".join(f"{k} ({n:,})" for k, n in gabung_bg.most_common(5)))
print("'virus corona':", f"{gabung_bg['virus corona']:,}")

Pasangan bigram tanpa combiner : 734,783
Pasangan bigram dengan combiner: 411,217 (1.8x lebih sedikit)
Selisih dengan pasangan kata H-9 (825,601): 90,818 = jumlah judul (90,818)
Bigram unik: 349,114

5 bigram teratas: virus corona (2,915), corona di (2,178), positif corona (1,804), di tengah (1,258), pandemi corona (1,075)
'virus corona': 2,915


**Penjelasan kode.** `mapper_bigram()` mengubah tiap judul menjadi pasangan kata berurutan (misalnya "virus corona") dengan `zip(tok, tok[1:])`. Jumlah pasangan dihitung tanpa combiner (semua kemunculan) dan dengan combiner (bigram unik per mapper), lalu dibandingkan dengan jumlah pasangan kata tunggal pada H-9. Jumlah bigram unik global juga dihitung.

**Interpretasi hasil.** Mapper bigram menghasilkan **734.783 pasangan** tanpa combiner dan **411.217** setelah combiner (hanya sekitar 1,8 kali lebih sedikit), dengan **349.114 bigram unik** secara global. Selisihnya terhadap pasangan kata tunggal H-9 (825.601) tepat **90.818, yaitu jumlah judul**, karena sebuah judul dengan n kata menghasilkan n-1 bigram. Penghematan combiner lebih kecil daripada kata tunggal (11 kali) karena kosakata bigram jauh lebih beragam, sehingga sedikit bigram yang berulang di dalam satu mapper. Lima bigram teratas: **virus corona (2.915)**, corona di (2.178), positif corona (1.804), di tengah (1.258), dan pandemi corona (1.075).

### N-4. Jumlah reducer: R = 2, 8, dan 32

In [ ]:
print(f"{'R':>4} {'terberat':>9} {'bagian adil':>12} {'rasio':>7}")
for r in (2, 8, 32):
    b = beban(hasil_map, r)
    terberat, adil = max(b) / sum(b) * 100, 100 / r
    print(f"{r:>4} {terberat:>8.2f}% {adil:>11.2f}% {terberat/adil:>6.2f}x")

   R  terberat  bagian adil   rasio
   2    52.27%       50.00%   1.05x
   8    16.83%       12.50%   1.35x
  32     5.82%        3.12%   1.86x


**Penjelasan kode.** H-11 dijalankan dengan R = 2, 8, dan 32 reducer, dan sel ini menghitung beban reducer terberat, bagian yang adil, serta rasio keduanya.

**Interpretasi hasil.** Tabel beban reducer terberat dibanding bagian yang adil (tanpa combiner):

| R | Terberat | Bagian adil | Rasio |
|---|---|---|---|
| 2 | 52,27% | 50,00% | 1,05x |
| 8 | 16,83% | 12,50% | 1,35x |
| 32 | 5,82% | 3,12% | 1,86x |

Bersama hasil H-11 (4 reducer: 1,17x; 16 reducer: 1,41x; 64 reducer: 2,74x), rasio ketimpangan **terus membesar** ketika reducer ditambah, karena beban terberat terkunci oleh key terbesar (`di` 3,32%) sementara bagian adil terus mengecil. Menambah reducer menurunkan beban absolut tiap reducer, tetapi tidak memperbaiki ketimpangan relatifnya.

### N-5. Crawl ulang dengan jeda

In [ ]:
t0 = time.perf_counter()
n_judul = sum(1 for _ in crawl_iter(BASE, range(1, 51), delay=0.05))
t_jeda = time.perf_counter() - t0

t0 = time.perf_counter()
sum(1 for _ in crawl_iter(BASE, range(1, 51), delay=0.0))
t_tanpa_jeda = time.perf_counter() - t0

kerja_per_halaman = t_tanpa_jeda / 50
estimasi = 911 * (kerja_per_halaman + 1.0)
print(f"50 halaman, jeda 0,05 detik : {t_jeda:.2f} detik ({n_judul:,} judul)")
print(f"50 halaman, tanpa jeda      : {t_tanpa_jeda:.2f} detik (kerja murni {kerja_per_halaman*1000:.1f} ms/halaman)")
print(f"Tambahan akibat jeda        : {t_jeda - t_tanpa_jeda:.2f} detik (teori 50 x 0,05 = 2,50 detik)")
print(f"Estimasi 911 halaman, jeda 1 detik: {estimasi:.0f} detik = {estimasi/60:.1f} menit")

50 halaman, jeda 0,05 detik : 5.00 detik (5,000 judul)
50 halaman, tanpa jeda      : 2.47 detik (kerja murni 49.3 ms/halaman)
Tambahan akibat jeda        : 2.54 detik (teori 50 x 0,05 = 2,50 detik)
Estimasi 911 halaman, jeda 1 detik: 956 detik = 15.9 menit


**Penjelasan kode.** Sel ini meng-crawl 50 halaman pertama dengan jeda 0,05 detik dan mencatat waktunya, lalu mengulang tanpa jeda untuk memperoleh waktu kerja murni per halaman. Estimasi 911 halaman dengan jeda 1 detik dihitung sebagai 911 × (1 detik + waktu kerja murni per halaman).

**Interpretasi hasil.** Crawl 50 halaman dengan jeda 0,05 detik selesai dalam **5,00 detik**, sedangkan crawl yang sama tanpa jeda selesai dalam **2,47 detik**. Selisih **2,54 detik** sangat dekat dengan nilai teoritis sebesar **50 × 0,05 = 2,50 detik**, sehingga dapat disimpulkan bahwa tambahan waktu hampir seluruhnya berasal dari jeda yang sengaja ditambahkan. Dengan waktu kerja murni sekitar **49,3 ms per halaman**, estimasi untuk **911 halaman dengan jeda 1 detik** adalah sekitar **956 detik** atau **15,9 menit**, dengan **911 detik** di antaranya berasal dari jeda dan sekitar **45 detik** merupakan waktu crawl murni.

### N-6. Sitemap lain

In [ ]:
SITEMAP_INDEKS = "https://www.detik.com/sitemap.xml"   # indeks sitemap detik.com (ubah bila alamatnya berbeda)

if not sampel:
    print("Hasil crawl H-4 kosong (tidak ada koneksi ke detik.com), latihan ini dilewati.")
else:
    try:
        sesi6 = requests.Session(); sesi6.headers["User-Agent"] = UA_DETIK
        indeks_sm = BeautifulSoup(ambil_xml(SITEMAP_INDEKS, sesi6), "xml")
        semua_loc = [l.get_text(strip=True) for l in indeks_sm.find_all("loc")]
        kandidat = [u for u in semua_loc if "sitemap_news.xml" in u and u not in SITEMAP_DIPILIH]
        print(f"Indeks sitemap: {len(semua_loc)} entri | sitemap berita selain yang dipakai: {len(kandidat)}")

        baru6, sm_baru = [], None
        for sm in kandidat[:3]:                      # maksimum 3 percobaan agar server tidak terbebani
            try:
                baru6 = parse_sitemap_news(ambil_xml(sm, sesi6))[:BATAS_PER_BAGIAN - 1]
                if baru6:
                    sm_baru = sm
                    break
            except Exception as e:
                print("  dilewati:", sm, "->", type(e).__name__)
        if not baru6:
            raise RuntimeError("tidak ada sitemap pengganti yang berisi judul")

        print("Sitemap pengganti 'sport':", sm_baru)
        hasil6 = ([x for x in sampel if x["kategori"] != "sport"] + baru6)[:BATAS_TOTAL]
        d6 = pd.DataFrame(hasil6)
        print("Kategori yang muncul:", d6["kategori"].value_counts().to_dict(), "| total:", len(d6))
        print("Kategori baru dari sitemap pengganti:", pd.DataFrame(baru6)["kategori"].value_counts().to_dict())
        print()
        for nama, lulus in aturan_kualitas(d6).items():
            print(f"{nama:<24} gagal: {(~lulus).sum():>3} ({(~lulus).mean()*100:5.1f}%)")
    except Exception as e:
        print("Latihan gagal:", type(e).__name__, "-", e)

Indeks sitemap: 536 entri | sitemap berita selain yang dipakai: 245
Sitemap pengganti 'sport': https://news.detik.com/suara-pembaca/sitemap_news.xml
Kategori yang muncul: {'news': 13, 'inet': 7} | total: 20
Kategori baru dari sitemap pengganti: {'news': 6}

judul tidak kosong       gagal:   0 (  0.0%)
judul 3-30 kata          gagal:   1 (  5.0%)
kategori dikenal         gagal:   0 (  0.0%)
tanggal valid            gagal:   0 (  0.0%)
url artikel valid        gagal:   0 (  0.0%)
judul tidak duplikat     gagal:   0 (  0.0%)


**Penjelasan kode.** Sel ini mengambil indeks sitemap detik.com, memilih sitemap berita yang belum dipakai (maksimum 3 percobaan, tiap request mematuhi `robots.txt` dan jeda 2 detik), lalu menggantikan sitemap `sport` pada `SITEMAP_DIPILIH`. Agar server tidak terbebani, hanya sitemap pengganti yang diambil (6 judul), digabung dengan judul `news` dan `inet` dari `sampel` H-4 sehingga total 20 judul. `aturan_kualitas()` dari H-6 diterapkan pada 20 judul itu. Bila crawl H-4 tidak berhasil, sel ini dilewati dengan pesan yang jelas.

**Interpretasi hasil.** Hasil bergantung pada indeks sitemap pada hari itu. Bila berhasil, tercetak jumlah entri indeks, alamat sitemap pengganti, sebaran kategori (kategori dari sitemap baru ikut muncul), dan jumlah baris gagal tiap aturan. Pada sesi Colab acuan, indeks `https://www.detik.com/sitemap.xml` dapat diakses dan sitemap berita pertama yang belum dipakai adalah bagian `suara-pembaca` dari news.detik.com, yang seluruh entrinya berkategori `news`, dan seluruh aturan kualitas bernilai **0 gagal**. Hasil 0 gagal wajar karena parser sitemap sudah mengekstrak judul, tanggal, dan URL artikel dengan benar. Aturan yang paling mungkin gagal adalah **kategori dikenal**, bila sitemap pengganti berasal dari kategori di luar sembilan kategori dataset (misalnya `wolipop`), yang menunjukkan bahwa aturan kualitas harus disesuaikan bila sumber data berganti. Bila alamat indeks pada `SITEMAP_INDEKS` berbeda, ubah variabel tersebut.

### N-7. Kata kunci

In [ ]:
data_kk = sampel if sampel else parse_sitemap_news(CONTOH_XML)
print("Sumber data:", "hasil crawl H-4" if sampel else "CONTOH_XML (hasil crawl tidak tersedia)", f"| {len(data_kk)} berita,",
      sum(1 for e in data_kk if e["kata_kunci"]), "di antaranya punya kata kunci")

def mapper_kk(entri):
    return [(k.strip().lower(), 1) for k in entri["kata_kunci"].split(",") if k.strip()]

pasangan_kk = [p for entri in data_kk for p in mapper_kk(entri)]
shuffle_kk = collections.defaultdict(list)
for k, n in pasangan_kk:
    shuffle_kk[k].append(n)

def reducer_kk(kunci, nilai):
    return (kunci, sum(nilai))

hasil_kk = dict(reducer_kk(k, v) for k, v in shuffle_kk.items())
print(f"Pasangan dari mapper: {len(pasangan_kk)} | kata kunci unik: {len(hasil_kk)} | jumlah reducer = pasangan: {sum(hasil_kk.values()) == len(pasangan_kk)}")
for k, n in sorted(hasil_kk.items(), key=lambda x: (-x[1], x[0]))[:10]:
    print(f"  {k:<28} {n}")

Sumber data: hasil crawl H-4 | 20 berita, 20 di antaranya punya kata kunci
Pasangan dari mapper: 86 | kata kunci unik: 68 | jumlah reducer = pasangan: True
  gibran rakabuming raka       4
  gibran rakabuming            3
  mpr                          3
  pengunduran diri             3
  basket                       2
  ibl all indonesian           2
  ibl all indonesian 2026      2
  jensen huang                 2
  operator telekomunikasi      2
  prabowo subianto             2


**Penjelasan kode.** Kata kunci tiap berita dihitung dengan gaya MapReduce: `mapper_kk()` memecah `kata_kunci` (dipisah koma) menjadi pasangan `(kata kunci, 1)`, shuffle mengelompokkan per key, dan `reducer_kk()` menjumlahkan. Data diambil dari hasil crawl H-4, atau dari `CONTOH_XML` bila crawl tidak tersedia. Kode juga memeriksa bahwa jumlah seluruh hitungan sama dengan jumlah pasangan dari mapper.

**Interpretasi hasil.** Dengan hasil crawl H-4, tercetak jumlah berita yang punya kata kunci, jumlah pasangan dari mapper, kata kunci unik, dan sepuluh kata kunci terbanyak. Isinya berubah setiap hari, sehingga tidak ada angka acuan tetap; pada sesi Colab acuan, 20 berita menghasilkan sekitar 93 sampai 96 pasangan dengan kata kunci terbanyak berupa nama tokoh atau topik hari itu (jumlah 2 sampai 3). Bila crawl tidak tersedia, data memakai `CONTOH_XML`: 3 berita, hanya entri pertama yang punya kata kunci, sehingga hasilnya **4 kata kunci unik** (`hukum`, `kepala daerah`, `mahkamah konstitusi`, `mk`) dengan jumlah masing-masing 1. Pemeriksaan jumlah pasangan sama dengan jumlah hitungan bernilai `True`, tanda tidak ada kata kunci yang hilang atau terhitung ganda.

# O. Tugas Mandiri: Inverted Index dengan MapReduce

Untuk setiap kata disimpan daftar id judul yang memuatnya. Mapper menghasilkan `(kata, id)` per partisi (dengan combiner lokal yang mengumpulkan id per kata), dan reducer menggabungkan daftar id dari semua mapper. **Penanganan id kosong pada kategori `inet`:** baris `inet` tidak dibuang (agar satu kategori utuh tidak hilang dari indeks) dan tidak diberi id acak. Setiap baris tanpa id diberi **id sintetis deterministik** `10^10 + CRC32(judul)`: nilainya jauh di atas id asli detik (jutaan), selalu sama setiap dijalankan, dan mudah dikenali sebagai id sintetis. Kata yang termasuk `STOPWORD` tidak diindeks.

In [ ]:
ID_SINTETIS = 10_000_000_000   # id >= 10^10 berarti id sintetis (baris inet tanpa url valid)

def id_dokumen(did, judul):
    return int(did) if not pd.isna(did) else ID_SINTETIS + zlib.crc32(judul.encode())

# Pemeriksaan: id sintetis harus unik
ids_inet = [id_dokumen(pd.NA, t) for t in bersih.loc[bersih["id"].isna(), "judul"]]
print("Baris tanpa id:", f"{len(ids_inet):,}", "| id sintetis unik:", f"{len(set(ids_inet)):,}")

def mapper_indeks(path):
    """Mapper + combiner: satu partisi menjadi {kata: [id, ...]} (id unik per kata per dokumen)."""
    d_ = pd.read_parquet(path, columns=["id", "judul"])
    lokal = collections.defaultdict(set)
    for did, t in zip(d_["id"], d_["judul"]):
        dok = id_dokumen(did, t)
        for kata in set(tokenisasi(t)):
            if kata not in STOPWORD:
                lokal[kata].add(dok)
    return {k: sorted(v) for k, v in lokal.items()}

def reducer_indeks(daftar_daftar_id):
    return sorted(set(i for daftar in daftar_daftar_id for i in daftar))

shuffle_idx = collections.defaultdict(list)
for hasil_m in map(mapper_indeks, splits):
    for kata, ids in hasil_m.items():
        shuffle_idx[kata].append(ids)
indeks = {kata: reducer_indeks(v) for kata, v in shuffle_idx.items()}
print("Kata dalam indeks:", f"{len(indeks):,}")

for kata in ("corona", "virus", "jokowi"):
    n_brute = sum(1 for t in bersih["judul"] if kata in set(tokenisasi(t)))
    print(f"\n'{kata}': {len(indeks[kata]):,} dokumen (hitung ulang tanpa MapReduce: {n_brute:,})")
    print("   lima id pertama:", indeks[kata][:5])

SIAP.mkdir(exist_ok=True)
pd.DataFrame({"kata": list(indeks), "jumlah_dokumen": [len(v) for v in indeks.values()],
              "id_dokumen": list(indeks.values())}).to_parquet(SIAP / "indeks_terbalik.parquet", index=False)
print("\nIndeks tersimpan di", SIAP / "indeks_terbalik.parquet")

Baris tanpa id: 5,577 | id sintetis unik: 5,577
Kata dalam indeks: 32,228

'corona': 17,228 dokumen (hitung ulang tanpa MapReduce: 17,228)
   lima id pertama: [4856495, 4856598, 4868317, 4868545, 4869163]

'virus': 3,067 dokumen (hitung ulang tanpa MapReduce: 3,067)
   lima id pertama: [4856495, 4856598, 4860201, 4865102, 4865341]

'jokowi': 2,316 dokumen (hitung ulang tanpa MapReduce: 2,316)
   lima id pertama: [4842674, 4842690, 4842703, 4842774, 4843215]

Indeks tersimpan di siap_pakai/indeks_terbalik.parquet


**Penjelasan kode.** Setiap baris diberi id dokumen: id asli dari URL, atau **id sintetis** `10^10 + CRC32(judul)` bila kosong (baris `inet`). Kode memeriksa bahwa id sintetis unik. `mapper_indeks()` memproses satu partisi: untuk tiap judul dikumpulkan himpunan kata unik (stopword dibuang) dan id dokumennya, lalu digabung per kata di sisi mapper (combiner lokal). `reducer_indeks()` menggabungkan daftar id dari semua mapper menjadi daftar terurut tanpa duplikat. Hasilnya diverifikasi dengan menghitung ulang jumlah dokumen tanpa MapReduce dan disimpan sebagai Parquet.

**Interpretasi hasil.** Terdapat **5.577 baris tanpa id** (seluruhnya `inet`) dan **5.577 id sintetis unik**, jadi tidak ada tabrakan; id sintetis selalu di atas 10.000.000.000, jauh di atas id asli detik (jutaan), sehingga mudah dibedakan dan berada di akhir daftar terurut. Indeks berisi **32.228 kata** (32.275 kata unik pada H-9 dikurangi kata yang termasuk stopword). Contoh: **corona** termuat di **17.228 dokumen** (sama dengan hitungan ulang tanpa MapReduce; angka ini sedikit di bawah 17.239 pada H-9 karena H-9 menghitung kemunculan, sedangkan indeks menghitung dokumen unik, sehingga judul yang memuat "corona" dua kali hanya terhitung sekali), lima id pertamanya `[4856495, 4856598, 4868317, 4868545, 4869163]`; **virus** di **3.067 dokumen** dengan lima id pertama `[4856495, 4856598, 4860201, 4865102, 4865341]`; dan **jokowi** di **2.316 dokumen** dengan lima id pertama `[4842674, 4842690, 4842703, 4842774, 4843215]`. **Alasan penanganan `inet`:** membuang barisnya menghilangkan 6,1% data dan satu kategori dari indeks, sedangkan id acak membuat indeks berubah setiap dijalankan; id sintetis berbasis CRC32 membuat baris `inet` tetap dapat dicari, deterministik, dan idempoten. Indeks tersimpan di `siap_pakai/indeks_terbalik.parquet`.

# P. Pertanyaan Evaluasi

### P-1. Tiga tahap MapReduce dan tahap perpindahan data

In [ ]:
map_k = [[map_fn(b) for b in k] for k in kasir]
dikirim_tanpa = sum(len(h) for h in map_k)
dikirim_dengan = sum(len(combiner(h)) for h in map_k)
print("MAP    : tiap kasir menghasilkan", [len(h) for h in map_k], "pasangan")
print("SHUFFLE: pasangan yang berpindah ke reducer =", dikirim_tanpa, "(tanpa combiner) /", dikirim_dengan, "(dengan combiner)")
print("REDUCE :", dict(reduce_fn(m, v) for m, v in {m: [j for h in map_k for mm, j in h if mm == m] for m in {m for h in map_k for m, _ in h}}.items()))

MAP    : tiap kasir menghasilkan [4, 4, 4] pasangan
SHUFFLE: pasangan yang berpindah ke reducer = 12 (tanpa combiner) / 9 (dengan combiner)
REDUCE : {'mie ayam': 4, 'es teh': 8, 'nasi goreng': 5}


**Penjelasan kode.** Kasus kantin dijalankan ulang dari `kasir` dengan fungsi H-8: map per kasir, hitung pasangan yang berpindah ke reducer tanpa dan dengan combiner, lalu reduce.

**Interpretasi hasil (jawaban).** MapReduce terdiri atas tiga tahap. **Map:** tiap kasir (mapper) memproses datanya sendiri dan menghasilkan pasangan `(menu, jumlah)`, yaitu 4 pasangan per kasir (12 total). **Shuffle:** semua pasangan dengan key yang sama dikumpulkan ke reducer yang sama. **Reduce:** tiap reducer menjumlahkan nilai per key sehingga hasilnya nasi goreng 5, es teh 8, mie ayam 4. Pada word count, mapper menghasilkan `(kata, 1)` dan reducer menjumlahkannya. **Data berpindah antar-mesin pada tahap shuffle:** 12 pasangan tanpa combiner dan 9 dengan combiner, sehingga shuffle adalah tahap paling mahal pada cluster.

### P-2. Fungsi combiner dan kapan tidak boleh dipakai

In [ ]:
# Combiner hanya aman bila operasinya asosiatif dan komutatif (jumlah, maks, min). Rata-rata TIDAK.
nilai_mapper = [[2, 4, 6], [10]]
rata_benar = sum(sum(x) for x in nilai_mapper) / sum(len(x) for x in nilai_mapper)
rata_combiner_salah = sum(sum(x) / len(x) for x in nilai_mapper) / len(nilai_mapper)   # rata-rata dari rata-rata
(s, n) = (sum(sum(x) for x in nilai_mapper), sum(len(x) for x in nilai_mapper))        # combiner yang benar: kirim (jumlah, banyak)
print("Rata-rata sebenarnya               :", rata_benar)
print("Combiner rata-rata (salah)         :", rata_combiner_salah)
print("Combiner mengirim (jumlah, banyak) :", s / n)

Rata-rata sebenarnya               : 5.5
Combiner rata-rata (salah)         : 7.0
Combiner mengirim (jumlah, banyak) : 5.5


**Penjelasan kode.** Sel ini menghitung rata-rata nilai `[2, 4, 6]` pada mapper pertama dan `[10]` pada mapper kedua dengan tiga cara: rata-rata sebenarnya, combiner yang menghitung rata-rata dari rata-rata (salah), dan combiner yang mengirim pasangan `(jumlah, banyak)` (benar).

**Interpretasi hasil (jawaban).** **Fungsi combiner** adalah reduce mini di sisi mapper sebelum shuffle sehingga data yang dikirim berkurang. Hasil akhir tidak berubah karena operasi penjumlahan bersifat **asosiatif dan komutatif**: menjumlahkan sebagian dulu lalu menjumlahkan hasilnya sama dengan menjumlahkan sekaligus (dibuktikan pada H-10: hasil identik `True` dengan pasangan turun dari 825.601 menjadi 74.009). **Combiner tidak boleh dipakai** bila operasi reduce tidak asosiatif atau komutatif, misalnya rata-rata: rata-rata sebenarnya **5,5**, tetapi rata-rata dari rata-rata mapper memberi **7,0** (salah karena bobot jumlah data tiap mapper berbeda). Solusinya combiner mengirim `(jumlah, banyak)` yang kemudian dibagi di reducer sehingga kembali **5,5**.

### P-3. Input split dan jumlah mapper

In [ ]:
print("Partisi kategori        :", len(splits), "berkas Parquet = 9 input split = 9 mapper")
print("Partisi bulan           :", bersih.groupby('bulan').ngroups, "berkas = 6 mapper bila dipakai")
print("Partisi kategori+bulan  :", bersih.groupby(['kategori', 'bulan']).ngroups, "berkas = 54 mapper bila dipakai")
print("Judul per split (kategori):", {p.parent.name.split('=')[1]: len(pd.read_parquet(p, columns=['judul'])) for p in splits})
print("\nPraktikum 3 memakai pola yang sama: Parquet dipartisi menurut borough_naik (K-10), sehingga satu direktori borough = satu input split.")
print("Jumlah mapper ditentukan oleh jumlah partisi (nilai kolom partisi), bukan oleh jumlah baris.")

Partisi kategori        : 9 berkas Parquet = 9 input split = 9 mapper
Partisi bulan           : 6 berkas = 6 mapper bila dipakai
Partisi kategori+bulan  : 54 berkas = 54 mapper bila dipakai
Judul per split (kategori): {'finance': 14141, 'food': 4314, 'health': 4914, 'hot': 16290, 'inet': 5577, 'news': 32321, 'oto': 4375, 'sport': 2433, 'travel': 6453}

Praktikum 3 memakai pola yang sama: Parquet dipartisi menurut borough_naik (K-10), sehingga satu direktori borough = satu input split.
Jumlah mapper ditentukan oleh jumlah partisi (nilai kolom partisi), bukan oleh jumlah baris.


**Penjelasan kode.** Sel ini mencetak jumlah berkas partisi untuk tiga skema (`kategori`, `bulan`, dan `kategori + bulan`) sebagai jumlah input split, jumlah judul pada tiap split, dan mengaitkannya dengan Praktikum 3.

**Interpretasi hasil (jawaban).** **Input split** adalah potongan data yang diproses oleh satu mapper secara independen; pada praktikum ini satu split adalah satu berkas partisi Parquet. Partisi `kategori` menghasilkan **9 berkas sehingga ada 9 mapper**, dengan ukuran split tidak seimbang (dari 2.433 judul pada `sport` hingga 32.321 judul pada `news`). Bila data dipartisi `bulan` jumlah mapper menjadi **6**, dan bila `kategori + bulan` menjadi **54**. **Pengaruh Praktikum 3:** di sana Parquet dipartisi menurut `borough_naik` (K-10), satu direktori per borough, sehingga bila dibaca MapReduce satu direktori borough menjadi satu input split. Jumlah mapper ditentukan oleh jumlah partisi (jumlah nilai kolom partisi), bukan oleh jumlah baris. Jadi cara kita mempartisi data menentukan cara pekerjaan dibagi: terlalu sedikit partisi membatasi paralelisme, terlalu banyak memunculkan small file problem.

### P-4. Data skew, contoh, dan dua cara mengurangi dampaknya

In [ ]:
# Cara 1: combiner (sudah terlihat pada H-11). Cara 2: salting, memecah key panas menjadi beberapa sub-key.
SALT = 8
panas = {k for k, _ in collections.Counter(hasil_semua).most_common(20)}

def pasangan_salting(pasangan_per_mapper):
    urut = 0
    for h in pasangan_per_mapper:
        for kata, n in h:
            yield (f"{kata}#{urut % SALT}" if kata in panas else kata), n
            urut += 1

for r in (16, 64):
    b0 = beban(hasil_map, r)
    b1 = [0] * r
    for key, n in pasangan_salting(hasil_map):
        b1[partitioner(key, r)] += 1
    print(f"{r:>3} reducer: terberat {max(b0)/sum(b0)*100:5.2f}% (biasa) -> {max(b1)/sum(b1)*100:5.2f}% (salting) | bagian adil {100/r:5.2f}%")

# Kebenaran: setelah tiap sub-key dijumlahkan lalu digabung kembali, hasilnya harus sama dengan H-9
sub = collections.Counter()
for key, n in pasangan_salting(hasil_map):
    sub[key] += n
gabung_lagi = collections.Counter()
for key, n in sub.items():
    gabung_lagi[key.split("#")[0]] += n
print("\nHasil akhir setelah sub-key digabung kembali sama dengan H-9:", dict(gabung_lagi) == hasil_semua)

 16 reducer: terberat  8.82% (biasa) ->  7.05% (salting) | bagian adil  6.25%
 64 reducer: terberat  4.28% (biasa) ->  2.25% (salting) | bagian adil  1.56%

Hasil akhir setelah sub-key digabung kembali sama dengan H-9: True


**Penjelasan kode.** Sel ini menunjukkan cara kedua mengurangi skew, yaitu **salting**: 20 kata paling sering dipecah menjadi 8 sub-key (misalnya `di#0` sampai `di#7`) yang disebar ke reducer berbeda. Beban reducer terberat dibandingkan dengan pembagian biasa untuk 16 dan 64 reducer. Kebenarannya diperiksa dengan menjumlahkan tiap sub-key lalu menggabungkannya kembali ke kata asal dan membandingkannya dengan hasil H-9.

**Interpretasi hasil (jawaban).** **Data skew** adalah beban yang tidak merata karena satu key sangat sering muncul sehingga reducer yang memegangnya menjadi bottleneck. **Contoh dari praktikum ini:** kata `di` sebanyak 27.390 pasangan (3,3% dari seluruh pasangan) membuat reducer terberat menerima 4,28% pada 64 reducer, padahal bagian adilnya 1,56%. **Dua cara mengurangi dampaknya:** (1) **combiner**, yang membuat tiap kata hanya dikirim sekali per mapper (`di` tinggal 9 pasangan; terberat pada 64 reducer turun menjadi 1,86%), dan (2) **salting**, yang memecah key panas menjadi beberapa sub-key sehingga beban tersebar: terberat turun dari 8,82% menjadi **7,05%** (16 reducer, adil 6,25%) dan dari 4,28% menjadi **2,25%** (64 reducer, adil 1,56%). Harga salting adalah satu langkah tambahan untuk menggabungkan sub-key, dan pemeriksaan membuktikan hasil akhir setelah digabung kembali **sama dengan H-9** (`True`). Menambah jumlah reducer saja tidak menyelesaikan skew dari satu key.

### P-5. Hadoop Streaming dan mapper bukan Java

In [ ]:
# Mapper yang sama sekali bukan Python (tr + sed): selama membaca stdin dan menulis kata<TAB>1, reducer.py tetap bekerja.
perintah = ("cut -f2 masukan.tsv | tr 'A-Z' 'a-z' | tr -cs 'a-z' '\\n' | sed '/^$/d' | sed 's/$/\\t1/' "
            "| LC_ALL=C sort | python3 reducer.py")
out = subprocess.run(perintah, shell=True, capture_output=True, text=True, executable="/bin/bash").stdout
hasil_shell = {k: int(n) for k, n in (b.split("\t") for b in out.strip().split("\n"))}
print("Kata unik dari mapper shell:", f"{len(hasil_shell):,}")
print("Sama dengan hasil mapper Python (H-9):", hasil_shell == hasil_semua)

Kata unik dari mapper shell: 32,275
Sama dengan hasil mapper Python (H-9): True


**Penjelasan kode.** Mapper Python diganti dengan mapper yang sama sekali bukan Python (`cut`, `tr`, dan `sed`) yang membaca teks dan menulis `kata<TAB>1`, lalu keluarannya diurutkan dengan `sort` dan dijumlahkan oleh `reducer.py` yang sama. Hasilnya dibandingkan dengan `hasil_semua` dari H-9.

**Interpretasi hasil (jawaban).** Hadoop Streaming memungkinkan mapper ditulis dalam Python (atau bahasa apa pun) karena komunikasinya hanya lewat **stdin dan stdout**: mapper membaca teks dari stdin dan mencetak `key<TAB>value` ke stdout, dan Hadoop tidak peduli bahasa pembuatnya. Buktinya, mapper berbasis `tr` dan `sed` menghasilkan **32.275 kata unik** dan hasil yang **sama persis** dengan mapper Python (`True`). **Kelebihannya:** cepat ditulis dan diuji (cukup dengan pipe Linux), kaya pustaka pengolah teks dan data, serta tim tidak perlu menguasai Java. **Kekurangannya:** data harus diserialisasi menjadi teks dan lewat proses terpisah sehingga lebih lambat dan tidak bisa memakai fitur lanjutan API Java (misalnya combiner dan partitioner kustom dengan tipe data asli), dan kesalahan parsing teks harus ditangani sendiri.

### P-6. Source, channel, sink, dan batas antrean

In [ ]:
def demo_flume(maks, n_event=300, delay_sink=0.002):
    q = queue.Queue(maxsize=maks)          # maks = 0 berarti tanpa batas
    stat = {"maks": 0, "menunggu": 0.0}
    def src_demo():
        for i in range(n_event):
            t0 = time.perf_counter()
            q.put(i)                        # menunggu bila antrean penuh
            stat["menunggu"] += time.perf_counter() - t0
            stat["maks"] = max(stat["maks"], q.qsize())
        q.put(None)
    def snk_demo():
        while q.get() is not None:
            time.sleep(delay_sink)          # sink lebih lambat daripada source
    a, b = threading.Thread(target=src_demo), threading.Thread(target=snk_demo)
    t0 = time.perf_counter(); a.start(); b.start(); a.join(); b.join()
    return stat["maks"], stat["menunggu"], time.perf_counter() - t0

for maks in (10, 0):
    m, tunggu, total_w = demo_flume(maks)
    nama = f"maxsize={maks}" if maks else "tanpa batas (maxsize=0)"
    print(f"{nama:<26} antrean maksimum = {m:>4} | waktu source menunggu = {tunggu:.3f} s | selesai {total_w:.2f} s")
print(f"\nPada H-5: channel 2.000 event, antrean maksimum yang tercapai = {statistik['max_antrean']:,} (batas tidak terlampaui: {statistik['max_antrean'] <= 2000})")

maxsize=10                 antrean maksimum =   10 | waktu source menunggu = 0.616 s | selesai 0.64 s
tanpa batas (maxsize=0)    antrean maksimum =  300 | waktu source menunggu = 0.001 s | selesai 0.64 s

Pada H-5: channel 2.000 event, antrean maksimum yang tercapai = 200 (batas tidak terlampaui: True)


**Penjelasan kode.** Sebuah source mengirim 300 event ke channel dengan sink yang sengaja lebih lambat (jeda 0,002 detik per event). Dua kondisi dibandingkan: channel berbatas (`maxsize=10`) dan channel tanpa batas (`maxsize=0`). Sel ini mencatat antrean maksimum dan total waktu source menunggu, lalu menampilkan antrean maksimum pada H-5.

**Interpretasi hasil (jawaban).** Pada Flume, **source** menerima data dari sumber, **channel** adalah antrean sementara yang menyangga kecepatan berbeda antara pengirim dan penulis, dan **sink** menulis data ke tujuan (pada praktikum ini: crawler, `queue.Queue`, dan penulis JSONL). Dengan batas 10, antrean maksimum tepat **10** dan source dipaksa **menunggu** (backpressure) hampir sepanjang demonstrasi (sekitar 0,6 dari 0,64 detik). Tanpa batas, antrean menumpuk hingga **300**, yaitu hampir semua event, dan source tidak pernah menunggu: memori tumbuh mengikuti data yang belum sempat ditulis. Itulah alasan channel membatasi ukuran antreannya: memori tetap terkendali walau source lebih cepat daripada sink. Pada H-5 batasnya 2.000 event dan antrean maksimum yang tercapai tidak melampauinya (`True`).

### P-7. Sqoop import dan export, mengapa memakai beberapa mapper dengan rentang kunci

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import sqlite3
DB = SIAP / "laporan.db"
c0 = sqlite3.connect(DB)
lo, hi = c0.execute("SELECT MIN(rowid), MAX(rowid) FROM kata_per_kategori").fetchone()
total_db = c0.execute("SELECT COUNT(*), SUM(jumlah) FROM kata_per_kategori").fetchone()
c0.close()

M = 4
langkah = (hi - lo + 1 + M - 1) // M
def mapper_import(i):
    a, b = lo + i * langkah, min(lo + (i + 1) * langkah - 1, hi)
    c = sqlite3.connect(DB)
    n, s = c.execute("SELECT COUNT(*), SUM(jumlah) FROM kata_per_kategori WHERE rowid BETWEEN ? AND ?", (a, b)).fetchone()
    c.close()
    return i, a, b, n, s

with ThreadPoolExecutor(M) as ex:
    hasil_imp = list(ex.map(mapper_import, range(M)))
for i, a, b, n, s in hasil_imp:
    print(f"mapper {i}: rowid {a:>6,} - {b:>6,} -> {n:>6,} baris")
print("Total dari 4 mapper:", sum(r[3] for r in hasil_imp), "baris | jumlah:", sum(r[4] for r in hasil_imp))
print("Total satu kueri   :", total_db[0], "baris | jumlah:", total_db[1])

mapper 0: rowid      1 - 18,401 -> 18,401 baris
mapper 1: rowid 18,402 - 36,802 -> 18,401 baris
mapper 2: rowid 36,803 - 55,203 -> 18,401 baris
mapper 3: rowid 55,204 - 73,601 -> 18,398 baris
Total dari 4 mapper: 73601 baris | jumlah: 709926
Total satu kueri   : 73601 baris | jumlah: 709926


**Penjelasan kode.** Sel ini mensimulasikan `sqoop import` dengan empat mapper paralel yang masing-masing membaca satu rentang `rowid` dari tabel `kata_per_kategori` pada `laporan.db`, lalu menjumlahkan hasilnya dan membandingkannya dengan satu kueri utuh.

**Interpretasi hasil (jawaban).** **Sqoop import** memindahkan data dari database relasional ke Hadoop, sedangkan **Sqoop export** sebaliknya, memindahkan hasil pemrosesan dari Hadoop ke database relasional (disimulasikan pada H-13 lewat `to_sql` ke `laporan.db`). Import memakai **beberapa mapper dengan rentang kunci** agar tabel besar dibaca paralel tanpa tumpang tindih: di sini empat mapper masing-masing membaca sekitar **18.401 baris** (rowid 1 sampai 18.401, 18.402 sampai 36.802, 36.803 sampai 55.203, dan 55.204 sampai 73.601). Totalnya **73.601 baris dengan jumlah 709.926**, **sama dengan satu kueri utuh**, jadi pembagian berdasarkan rentang kunci tidak menghilangkan maupun menggandakan data, dan tiap mapper dapat dijalankan di mesin berbeda.

### P-8. Peran HDFS dan YARN

In [ ]:
ukuran_masukan = os.path.getsize("masukan.tsv")
for blok_mb in (128, 1):
    n_blok = -(-ukuran_masukan // (blok_mb * 1024 * 1024))
    print(f"masukan.tsv = {ukuran_masukan/1e6:.2f} MB -> blok {blok_mb} MB: {n_blok} blok (= {n_blok} input split bila 1 split per blok)")
print("lapisan_terkurasi/ : 9 berkas partisi berperan sebagai blok (input split kita), masing-masing di bawah 4 MB:",
      [round(p.stat().st_size / 1e6, 2) for p in splits])
print(f"\n'Slot YARN' pada H-10: Pool(2) dengan CPU tersedia = {os.cpu_count()} (container disimulasikan oleh proses Python)")
print("Direktori lapisan ('HDFS' lokal):", sorted(p.name for p in Path('.').iterdir() if p.is_dir() and p.name.startswith(('lapisan', 'siap'))))

masukan.tsv = 6.08 MB -> blok 128 MB: 1 blok (= 1 input split bila 1 split per blok)
masukan.tsv = 6.08 MB -> blok 1 MB: 6 blok (= 6 input split bila 1 split per blok)
lapisan_terkurasi/ : 9 berkas partisi berperan sebagai blok (input split kita), masing-masing di bawah 4 MB: [1.24, 0.45, 0.46, 1.57, 0.21, 3.22, 0.41, 0.22, 0.58]

'Slot YARN' pada H-10: Pool(2) dengan CPU tersedia = 2 (container disimulasikan oleh proses Python)
Direktori lapisan ('HDFS' lokal): ['lapisan_mentah', 'lapisan_terkurasi', 'siap_pakai']


**Penjelasan kode.** Sel ini menghitung berapa blok HDFS yang diperlukan `masukan.tsv` untuk ukuran blok 128 MB (bawaan) dan 1 MB, menampilkan ukuran berkas partisi (pengganti blok), dan menunjukkan sumber daya yang diatur `Pool` (pengganti YARN).

**Interpretasi hasil (jawaban).** **HDFS** menyimpan berkas besar yang dipecah menjadi blok (128 MB) dan direplikasi di banyak mesin. **YARN** mengatur sumber daya (CPU dan memori) di cluster dan menjalankan job; MapReduce hanyalah salah satu program di atasnya. Pada praktikum ini keduanya **hanya disimulasikan**. `masukan.tsv` berukuran sekitar 6,08 MB hanya memerlukan **1 blok** pada ukuran 128 MB, dan baru menjadi **6 blok** bila ukuran blok 1 MB, sehingga data sekecil ini tidak memerlukan HDFS. Berkas partisi Parquet di disk lokal berperan seperti blok (kesembilannya di bawah 4 MB, terbesar sekitar 3 MB untuk `news`), dan `multiprocessing.Pool(2)` berperan seperti YARN yang membagi CPU komputer ke dua mapper. Keduanya terlihat sungguhan hanya bila bagian **opsional H-12** dijalankan pada Hadoop (itupun *local mode* tanpa HDFS dan YARN), atau pada cluster yang sebenarnya.

## Lampiran. Pengemasan Keluaran untuk Dikumpulkan

folder `lapisan_mentah/`, `lapisan_terkurasi/`, dan `siap_pakai/` dikumpulkan dalam bentuk zip, tanpa data mentah bila berukuran besar (cukup contoh). Sel berikut mengemas dua berkas batch pertama dari `lapisan_mentah/` sebagai contoh, seluruh `lapisan_terkurasi/` dan `siap_pakai/`, serta `contoh_crawl/` bila ada.

In [ ]:
import zipfile
NAMA_ZIP = "BD_P05_2411531002_folder_hasil.zip"
batch_contoh = sorted(Path("lapisan_mentah").glob("*.jsonl"))[:2]
berkas = batch_contoh + [p for f in ("lapisan_terkurasi", "siap_pakai", "contoh_crawl") if Path(f).exists() for p in sorted(Path(f).rglob("*")) if p.is_file()]
with zipfile.ZipFile(NAMA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in berkas:
        z.write(p)
with zipfile.ZipFile(NAMA_ZIP) as z:
    nama = z.namelist()
print("Berkas zip      :", Path(NAMA_ZIP).resolve())
print("Jumlah berkas   :", len(nama), "| ukuran zip:", round(os.path.getsize(NAMA_ZIP) / 1e6, 1), "MB")
print("Isi per folder  :", {f: sum(1 for n in nama if n.startswith(f + "/")) for f in ("lapisan_mentah", "lapisan_terkurasi", "siap_pakai", "contoh_crawl")})

Berkas zip      : /content/drive/MyDrive/BigData/Praktikum5/BD_P05_2411531002_folder_hasil.zip
Jumlah berkas   : 16 | ukuran zip: 18.7 MB
Isi per folder  : {'lapisan_mentah': 2, 'lapisan_terkurasi': 9, 'siap_pakai': 4, 'contoh_crawl': 1}
